# TokaMark `task_1-3` Sensor-Robustness Study (Gupta arXiv 2607.11915) on Kaggle

**Goal:** Run the complete sensor-failure robustness evaluation on a *trained, frozen* task_1-3 model across 6 failure scenarios (random dropout, channel ablation, category importance, temporal gap, correlated group failure, disruption-proximate) and 3 imputation modes (zero, train-mean, forward-fill).

**Requirements on Kaggle:**
- **Accelerator:** GPU T4 (x1 or x2).
- **Internet:** ON (for downloading test shots from Hugging Face).
- **Input Data Attached:** Add the output of your training notebook (`Fusion_recon` or `run_40gb`) to `/kaggle/input/`.
- **Execution:** Click **"Save Version" -> "Save & Run All (Commit)"** to run headlessly in the background (~25-35 minutes total).


In [ ]:
import os, sys, time, shutil, subprocess, json, glob

# ================= CONFIG =================
DATA_SPLIT    = "random"  # "random" or "temporal"
MODEL_TYPE    = "lstm"    # trained failure-aware CNN+LSTM
FAILURE_AWARE = 1        # must match the trained model: 1 = Module 2 Failure-Aware + PINN checkpoint, 0 = baseline
BATCH         = 512       # batch size for test inference
SEED          = 42
DL_WORKERS    = 0         # 0 = main process (hang-proof on Kaggle)
ROB_SCENARIOS = "all"     # or comma-separated list of scenarios
ROB_IMPUTE    = "zero,mean,ffill"
# ==========================================

T0 = time.time()
os.environ["TM_T0"] = str(T0)

def run(cmd, check=True):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=os.environ)
    for line in p.stdout: print(line, end="", flush=True)
    p.wait()
    if check and p.returncode: raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p.returncode

print("=== System Resources ===")
run("nvidia-smi -L; nproc; free -g | head -2; df -h /kaggle/working /tmp 2>/dev/null", check=False)
run("mount -o remount,size=8G /dev/shm 2>/dev/null || sudo mount -o remount,size=8G /dev/shm", check=False)
run("df -h /dev/shm", check=False)

W = "/kaggle/working"
cands = [d for d in ("/kaggle/temp", "/tmp", "/kaggle/working") if os.path.isdir(d)]
DATA_ROOT = max(cands, key=lambda d: shutil.disk_usage(d).free) + "/tokamark_data"
print(f"data root: {DATA_ROOT} | free GB: {shutil.disk_usage(os.path.dirname(DATA_ROOT)).free/1e9:.2f}")

# Install necessary runtime dependencies
run("python -m pip install -q zarr==3.1.5 xarray 'fsspec==2025.3.0' 's3fs==2025.3.0' "
    "huggingface_hub pyyaml scipy requests filelock torchinfo==1.8.0", check=True)
run("python -c \"import zarr, xarray, s3fs, huggingface_hub, torchinfo, requests, filelock; print('dependency imports OK')\"", check=True)

# ==============================================================================
# RESTORE TRAINED ARTIFACTS FROM /kaggle/input (Handles DATASETS and NOTEBOOKS)
# ==============================================================================
print("\n=== Checking /kaggle/input for trained model artifacts ===")
input_dirs = glob.glob("/kaggle/input/*")
print("Input folders detected:", [os.path.basename(d) for d in input_dirs])

restored = False

# Priority 1: Direct match in dataset or notebook folders (e.g. /kaggle/input/results/out)
for cand in input_dirs:
    cand_out = os.path.join(cand, "out")
    if os.path.isdir(cand_out):
        print(f"Found artifacts in input directory: {cand}")
        run(f"mkdir -p '{W}/out' && cp -r '{cand_out}/'* '{W}/out/'")
        for r in ("tokamark", "tokamark_baseline"):
            r_src = os.path.join(cand, r)
            if os.path.isdir(r_src) and not os.path.isdir(f"{W}/{r}"):
                print(f"Restoring repo {r} from {cand}...")
                run(f"cp -r '{r_src}' '{W}/'")
        restored = True
        break

# Priority 2: Check for any results.zip archive in /kaggle/input
if not restored:
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f.endswith(".zip") and any(k in f.lower() for k in ("result", "fusion", "run", "40gb")):
                fpath = os.path.join(root, f)
                print(f"Extracting zip archive: {fpath} -> {W}/")
                run(f"unzip -q -o '{fpath}' -d {W}/")
                restored = True
                break
        if restored:
            break

# Priority 3: Deep recursive search for best_model.pt
if not restored:
    for root, dirs, files in os.walk("/kaggle/input"):
        if "best_model.pt" in files:
            cur = root
            while cur and cur != "/kaggle/input":
                if os.path.basename(cur) == "out":
                    print(f"Found out directory at {cur}, copying to {W}/out...")
                    run(f"mkdir -p '{W}/out' && cp -r '{cur}/'* '{W}/out/'")
                    restored = True
                    break
                cur = os.path.dirname(cur)
            if restored:
                break

# Verify and position the restored model checkpoint
expected_pt = f"{W}/out/{DATA_SPLIT}/model/best_model.pt"
if not os.path.exists(expected_pt):
    for root, dirs, files in os.walk(W):
        if "best_model.pt" in files:
            src_pt = os.path.join(root, "best_model.pt")
            run(f"mkdir -p '{W}/out/{DATA_SPLIT}/model' && cp '{src_pt}' '{expected_pt}'")
            print(f"[OK] Located and positioned model checkpoint to: {expected_pt}")
            break

if not os.path.exists(expected_pt):
    raise RuntimeError("Missing best_model.pt! Please attach your trained model results archive to /kaggle/input.")

print(f"[OK] Model successfully verified at: {expected_pt}")
expected_sel = f"{W}/out/{DATA_SPLIT}/selection.json"
if os.path.exists(expected_sel):
    print(f"[OK] Selection file verified at: {expected_sel}")
else:
    print(f"[INFO] selection.json not in archive - Step 1 (prep) will download official {DATA_SPLIT} test split automatically.")

# Ensure repos exist (clone if not copied from input)
REPOS = (("tokamark", "1a200f4"), ("tokamark_baseline", "1e7c489"))
for name, sha in REPOS:
    if not os.path.isdir(f"{W}/{name}"):
        run(f"git clone -q https://github.com/UKAEA-IBM-STFC-Fusion-FMs/{name}.git {W}/{name}")
    run(f"cd {W}/{name} && git checkout -q {sha} && git log -1 --format='{name} @ %h'")

env = dict(
    TM_SEED=SEED,
    TM_DATA_ROOT=DATA_ROOT,
    TM_WORK=W,
    TM_TOKAMARK=f"{W}/tokamark",
    TM_BASELINE=f"{W}/tokamark_baseline",
    TM_DL_WORKERS=DL_WORKERS,
    TM_GIT_REVS=",".join(f"{n}@{s}" for n, s in REPOS),
    TM_BATCH=BATCH,
    TM_MODEL=MODEL_TYPE, TM_FAILURE_AWARE=("1" if FAILURE_AWARE else "0"),
    TM_ROB_SCENARIOS=ROB_SCENARIOS,
    TM_ROB_IMPUTE=ROB_IMPUTE,
    TM_ROB_SEED=SEED,
    TM_ROB_BATCH=BATCH,
)
os.environ.update({k: str(v) for k, v in env.items()})
print("Environment configured successfully.")


In [ ]:
%%writefile /kaggle/working/pipeline.py
#!/usr/bin/env python
"""TokaMark task_1-3 CNN baseline on Kaggle: prep | train | eval | plot | summary.

Usage: python pipeline.py {prep,train,eval,plot,summary} [--split random|temporal]
All settings come from environment variables (see CFG below), set by the notebook.

Protocol notes (deviations from the official run_training.py / run_evaluation.py):
- Custom step-based training loop mirroring src/trainer.py:BatchStepTrainer (same
  Adam(lr, wd=1e-4) + MultiOutputMSELoss, validate_every, patience, drop_last=True),
  extended with: wall-clock deadline, periodic + atomic checkpointing, resume from
  last.pt, CUDA-OOM reporting, and non-finite-loss skipping.
- Shots are a random subset OF THE OFFICIAL SPLIT LISTS (membership never mixed),
  chosen under a byte budget; official stats/outlier/split files always match the
  split protocol. test uses shuffle_windows=False (official eval behaviour).
"""
import os, sys, json, time, math, random, shutil, zipfile, argparse, subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

T0 = float(os.environ.get("TM_T0", time.time()))  # session start (set by notebook)


def env(k, d):
    return type(d)(os.environ.get(k, d))


CFG = dict(
    task=env("TM_TASK", "task_1-3"),
    model=env("TM_MODEL", "mlp"),                         # mlp (CNN+MLP) or lstm (CNN+LSTM)
    seed=env("TM_SEED", 23),
    repo_id="UKAEA-IBM-STFC/tokamark-v1",
    work=env("TM_WORK", "/kaggle/working"),
    data_root=env("TM_DATA_ROOT", "/tmp/tokamark_data"),
    tokamark_dir=env("TM_TOKAMARK", "/kaggle/working/tokamark"),
    baseline_dir=env("TM_BASELINE", "/kaggle/working/tokamark_baseline"),
    data_budget_gb=env("TM_DATA_GB", 40.0),
    prep_deadline_h=env("TM_PREP_DEADLINE_H", 1.75),      # hours since T0: stop downloading
    train_deadline_h=env("TM_TRAIN_DEADLINE_H", 8.0),     # hours since T0: stop training
    batch_size=env("TM_BATCH", 512),
    validate_every=env("TM_VALIDATE_EVERY", 100),
    patience=env("TM_PATIENCE", 10),
    lr=1e-4,
    weight_decay=env("TM_WEIGHT_DECAY", 1e-4),            # official Adam weight decay
    train_from_scratch=env("TM_TRAIN_FROM_SCRATCH", "0") == "1",
    max_steps=env("TM_MAX_STEPS", 10_000_000),
    dl_workers=env("TM_DL_WORKERS", -1),                  # -1 -> auto; 0 -> main process only; N -> N workers
    use_dp=env("TM_USE_DP", "auto"),                      # auto|0|1
    ckpt_every_s=env("TM_CKPT_EVERY_S", 600),
    force_reselect=env("TM_FORCE_RESELECT", "0") == "1",
    plot_windows=env("TM_PLOT_WINDOWS", 3),
    git_revs=env("TM_GIT_REVS", "unknown"),
    index_threads=env("TM_INDEX_THREADS", 12),            # concurrency; global req/s throttle (_RL_RPS) is the real limiter
    index_stop_gb=env("TM_INDEX_STOP_GB", 0.0),           # 0 -> auto: max(3, 1.6 x data budget)
    index_min_tars=env("TM_INDEX_MIN_TARS", 6),           # spread subset across >= this many archives
    failure_aware=env("TM_FAILURE_AWARE", "0") == "1",    # 1 -> Module 2 failure-aware masked architecture
    fail_aug_p=env("TM_FAIL_AUG_P", 0.5),                 # probability of failure augmentation during training
    pinn_lambda=env("TM_PINN_LAMBDA", 0.1),               # weight of flux-loop consistency loss
)


def log(*a):
    print(f"[{(time.time() - T0) / 60:7.1f}m]", *a, flush=True)


def paths(split):
    out = os.path.join(CFG["work"], "out", split)
    return dict(
        out=out,
        data=os.path.join(CFG["data_root"], split),
        index=os.path.join(CFG["work"], "out", "tar_index.json"),
        sel=os.path.join(out, "selection.json"),
        model_dir=os.path.join(out, "model") + "/",
        results=os.path.join(out, "results") + "/",
        figures=os.path.join(out, "figures") + "/",
    )


def setup_imports():
    for p in (os.path.join(CFG["tokamark_dir"], "src"), CFG["baseline_dir"]):
        if p not in sys.path:
            sys.path.insert(0, p)
    os.chdir(CFG["baseline_dir"])


def split_files(split):
    """Official split CSV + stats + outlier file; ALWAYS matched per protocol."""
    setup_imports()
    from tokamark.tools.path import (RANDOM_SPLIT_TOKAMARK_DATA_SPLITS_FILE as RS, RANDOM_SPLIT_SIGNALS_STATS_FILE as RT,
                                     TEMPORAL_SPLIT_TOKAMARK_DATA_SPLITS_FILE as TS, TEMPORAL_SPLIT_SIGNALS_STATS_FILE as TT)
    from MAST_tools.utils.path_utils import RANDOM_SPLIT_OUTLIER_METADATA_FILE as RO, TEMPORAL_SPLIT_OUTLIER_METADATA_FILE as TO
    if split == "random":
        return RS, RT, RO
    if split == "temporal":
        return TS, TT, TO
    raise ValueError(f"unknown split {split}")


def patch_stats_module(stats):
    """Point get_signals_metadata at the stats file of the CURRENT split protocol.

    Upstream get_task_metadata() always reads the RANDOM-split stats file, and
    tokamark.evaluator binds get_signals_metadata at import time; patch both module
    references so metadata and NRMSE normalisation always match the protocol.
    """
    setup_imports()
    import tokamark.tasks as _tt
    import tokamark.evaluator as _te
    if getattr(_tt, "_tm_patched", None) == stats:
        return
    orig = getattr(_tt, "_tm_orig", None) or _tt.get_signals_metadata
    _tt._tm_orig = orig
    patched = lambda file_path=stats, _o=orig: _o(file_path=file_path)  # noqa: E731
    _tt.get_signals_metadata = patched
    _te.get_signals_metadata = patched
    _tt._tm_patched = stats
    log(f"patched get_signals_metadata -> {os.path.basename(stats)}")


def first_batch(loader, tries=100):
    """Return the first non-empty batch (collate_fn may yield None for empty batches)."""
    it = iter(loader)
    for _ in range(tries):
        b = next(it, None)
        if b is not None:
            return b
    raise RuntimeError("loader produced no non-empty batch")


def first_batch_timed(loader, tag="first batch", tries=100, timeout_s=1800):
    """first_batch with a heartbeat + hard wall-clock cap.

    A silent hang here (workers stuck on IPC / fork-after-CUDA / slow zarr reads) used
    to look like training forever; now it is visible every 30 s and fails loudly.
    """
    import threading
    res, done = {}, threading.Event()

    def _work():
        try:
            res["b"] = first_batch(loader, tries)
        except BaseException as e:  # noqa
            res["e"] = e
        finally:
            done.set()

    threading.Thread(target=_work, daemon=True).start()
    t0 = time.time()
    while not done.wait(30):
        log(f"  waiting for {tag}... {time.time() - t0:.0f}s (workers are still producing it)")
        if time.time() - t0 > timeout_s:
            raise RuntimeError(
                f"{tag} did not arrive within {timeout_s}s. This is a dataloader hang, not training. "
                f"Try TM_DL_WORKERS=0 (main-process loading, no workers/IPC) or a smaller TM_BATCH "
                f"({CFG['batch_size']} -> 128/64).")
    if "e" in res:
        raise res["e"]
    return res["b"]


# ======================================================================================
# PREP: index tars with range requests, select shots per official split, fetch members
# ======================================================================================
def hf_headers():
    tok = os.environ.get("HF_TOKEN")
    return {"Authorization": f"Bearer {tok}"} if tok else {}


import threading
_TLS = threading.local()


def _session():
    """Per-thread requests.Session: TLS keep-alive turns ~1-3 s/request into ~0.2 s."""
    s = getattr(_TLS, "session", None)
    if s is None:
        import requests
        s = _TLS.session = requests.Session()
    return s


class _Deadline(Exception):
    pass


_RL_LOCK = threading.Lock()
_RL_NEXT = [0.0]        # earliest wall-clock time the next request may start (shared throttle)
_RL_RPS = [float(os.environ.get("TM_HF_RPS", 8))]    # current allowed req/s (adaptive)
_RL_RPS_MIN = max(1.0, float(os.environ.get("TM_HF_RPS_MIN", 2)))    # floor when throttled
_RL_429 = [0]           # consecutive-ish 429 counter (decays on success)


def _throttle():
    """Block until this thread may issue the next HF range request.

    All threads share one token budget (req/s). The rate is ADAPTIVE: each 429
    halves it (down to _RL_RPS_MIN) and every ~50 clean requests nudges it back up
    toward the configured ceiling. So an over-aggressive TM_HF_RPS self-corrects
    instead of failing — the setting sets the ceiling, the server decides the pace.
    """
    with _RL_LOCK:
        now = time.time()
        wait = _RL_NEXT[0] - now
        if wait > 0:
            time.sleep(wait)
        _RL_NEXT[0] = max(now, _RL_NEXT[0]) + 1.0 / max(0.5, _RL_RPS[0])


def _throttle_on_429():
    """Called on HTTP 429: halve the shared rate (floor _RL_RPS_MIN) so all threads slow."""
    with _RL_LOCK:
        _RL_429[0] += 1
        _RL_RPS[0] = max(_RL_RPS_MIN, _RL_RPS[0] / 2.0)
        return _RL_RPS[0]


def _throttle_on_ok():
    """Called on success: slowly recover toward the configured ceiling."""
    with _RL_LOCK:
        _RL_429[0] = max(0, _RL_429[0] - 1)
        if _RL_429[0] == 0:
            ceiling = float(os.environ.get("TM_HF_RPS", 8))
            _RL_RPS[0] = min(ceiling, _RL_RPS[0] * 1.05)


def ranged_get(url, a, b, tries=10, deadline=None):
    last = None
    for i in range(tries):
        if deadline is not None and time.time() > deadline:
            raise _Deadline()
        _throttle()
        try:
            r = _session().get(url, headers={**hf_headers(), "Range": f"bytes={a}-{b}"}, timeout=60)
            if r.status_code in (200, 206):
                if r.status_code == 200 and len(r.content) > (b - a + 1):  # server ignored Range
                    raise RuntimeError("Range ignored by server")
                _throttle_on_ok()
                return r.content
            last = f"HTTP {r.status_code}"
            if r.status_code == 429:
                # 429 = Too Many Requests. Shrink the shared rate (adaptive), honour
                # Retry-After, and push the shared clock forward so siblings slow too.
                rate = _throttle_on_429()
                ra = r.headers.get("Retry-After")
                try:
                    pause = float(ra) if ra is not None else 0.0
                except ValueError:
                    pause = 0.0
                pause = max(pause, min(120.0, 3.0 * (2 ** i)))
                with _RL_LOCK:
                    _RL_NEXT[0] = max(_RL_NEXT[0], time.time() + pause)
                time.sleep(pause + random.random() * 2.0)
                if i == 2:
                    log(f"  HF rate-limiting (429): shared rate -> {rate:.1f} req/s (auto-adapting)")
                continue
            if r.status_code in (500, 502, 503, 504):
                time.sleep(min(60.0, 2.0 * (2 ** i)) + random.random())
                continue
            time.sleep(min(30.0, 1.0 * (2 ** i)))
        except _Deadline:
            raise
        except Exception as e:  # noqa
            last = repr(e)
            time.sleep(min(30.0, 1.0 * (2 ** i)) + random.random())
    raise RuntimeError(f"range {a}-{b} failed after {tries} tries: {last}")


def make_ranged_reader(url, total, deadline=None, span=4096):
    """Small-read-optimised reader for parse_tar_headers.

    Header walks issue many 512-byte reads a few KB apart; HF resolve latency makes
    one-request-per-read prohibitively slow, so each read pulls a `span`-byte window
    and serves nearby reads from a one-window cache.
    """
    buf = {"start": -1, "data": b""}

    def read(a, n):
        end = min(a + n, total)
        if a >= total or end <= a:
            return b""
        if buf["start"] <= a and end <= buf["start"] + len(buf["data"]):
            o = a - buf["start"]
            return buf["data"][o:o + (end - a)]
        b = min(a + max(span, n) - 1, total - 1)
        data = ranged_get(url, a, b, deadline=deadline)
        buf["start"], buf["data"] = a, data
        return data[: end - a]

    return read


def _octal_or_b256(field):
    if field[0] & 0x80:
        return int.from_bytes(field, "big") & ((1 << (8 * len(field) - 1)) - 1)
    s = field.split(b"\0")[0].strip()
    return int(s, 8) if s else 0


def parse_tar_headers(read, total):
    """Walk a tar using read(offset, length)->bytes. Returns [(name, data_offset, size)]."""
    members, off, pending = [], 0, None
    while off + 512 <= total:
        h = read(off, 512)
        if len(h) < 512 or h.count(0) == 512:
            break
        name = h[:100].split(b"\0")[0].decode("utf8", "replace")
        if h[257:262] == b"ustar" and h[345:500].strip(b"\0"):
            name = h[345:500].split(b"\0")[0].decode("utf8", "replace") + "/" + name
        size = _octal_or_b256(h[124:136])
        flag = h[156:157]
        data_off = off + 512
        nxt = data_off + ((size + 511) // 512) * 512
        if flag in (b"L", b"x", b"g", b"K"):
            data = read(data_off, size)
            if flag == b"L":
                pending = data.split(b"\0")[0].decode("utf8", "replace")
            elif flag == b"x":
                for rec in data.split(b"\n"):
                    if b" path=" in rec:
                        pending = rec.split(b" path=", 1)[1].decode("utf8", "replace")
            off = nxt
            continue
        if flag in (b"0", b"\0"):
            members.append((pending or name, data_off, size))
        pending = None
        off = nxt
    return members


def build_index(deadline_ts, hard_deadline=None):
    """Index HF tars via range-request header walks. Resumable (index saved per tar).

    Tars are walked in a seeded-shuffled order and indexing stops early once the
    indexed shot bytes cover the download budget (x1.6 margin) - we only need enough
    archives to draw the subset, not the whole 500+ GB repo.

    `deadline_ts` is a SOFT deadline: once it passes, in-flight walks are cancelled
    when we already have enough (stop_reached) OR enough tars to draw a valid subset;
    otherwise indexing continues up to `hard_deadline` (the prep deadline), because an
    under-sized index risks the fatal <5-shot guard. Re-runs continue from the index.
    """
    hard_deadline = hard_deadline or deadline_ts
    from huggingface_hub import HfApi, hf_hub_url
    P = paths("random")
    os.makedirs(os.path.dirname(P["index"]), exist_ok=True)
    idx = json.load(open(P["index"])) if os.path.exists(P["index"]) else {"tars": {}, "shots": {}}
    api = HfApi()
    files = [f for f in api.list_repo_tree(CFG["repo_id"], repo_type="dataset", recursive=True)
             if getattr(f, "size", None) is not None and f.path.endswith(".tar")]
    files.sort(key=lambda f: f.path)
    log(f"HF repo lists {len(files)} tar files, {sum(f.size for f in files) / 1e9:.1f} GB total")
    indexed_bytes = sum(v[2] for v in idx["shots"].values())
    stop_gb = CFG["index_stop_gb"] or max(3.0, 1.6 * CFG["data_budget_gb"])
    min_tars = max(1, CFG["index_min_tars"])
    log(f"index: {len(idx['shots'])} shots / {indexed_bytes / 1e9:.1f} GB indexed (cached) across {len(idx['tars'])} tars; "
        f"stop submitting beyond {stop_gb:.1f} GB and >= {min_tars} tars")
    todo = [f for f in files if f.path not in idx["tars"]]
    random.Random(CFG["seed"]).shuffle(todo)  # seeded spread across the archive

    def stop_reached():
        return indexed_bytes >= stop_gb * 1e9 and len(idx["tars"]) >= min_tars

    def usable():
        # enough to select a valid subset even if a few tars still fail
        return len(idx["tars"]) >= min(3, min_tars)

    def work(f):
        url = hf_hub_url(CFG["repo_id"], f.path, repo_type="dataset")
        # reads may run until the HARD deadline; the loop cancels in-flight walks at the
        # soft deadline when the index is already usable.
        read = make_ranged_reader(url, f.size, deadline=hard_deadline)
        nreq = [0]
        mem = parse_tar_headers(lambda a, n: (nreq.__setitem__(0, nreq[0] + 1), read(a, n))[1], f.size)
        return f.path, url, mem

    with ThreadPoolExecutor(CFG["index_threads"]) as ex:
        it = iter(todo)
        pending = {}
        retries = {}
        inflight = set()

        def submit():
            if stop_reached():
                return False
            try:
                f = next(it)
            except StopIteration:
                # second pass: re-queue tars that failed transiently (e.g. HTTP 429),
                # never ones still being walked in another thread.
                cand = [f for f in todo
                        if f.path not in idx["tars"] and f.path not in inflight and retries.get(f.path, 0) < 3]
                if not cand:
                    return False
                f = cand[0]
                retries[f.path] = retries.get(f.path, 0) + 1
            fu = ex.submit(work, f)
            pending[fu] = f.path
            inflight.add(f.path)
            return True

        need = stop_gb * 1e9 - indexed_bytes
        cum, n_need = 0, 0
        if need > 0:
            for f in todo:
                cum += f.size
                n_need += 1
                if cum >= need:
                    break
        min_tars_left = max(0, min_tars - len(idx["tars"]))
        initial = min(CFG["index_threads"], max(n_need, min_tars_left), len(todo))
        log(f"submitting {initial} tars now (need ~{n_need} for budget, {min_tars_left} for archive spread)")
        for _ in range(initial):
            if not submit():
                break
        last_tick = time.time()
        while pending:
            now = time.time()
            if now > hard_deadline:
                log(f"index HARD deadline reached ({len(idx['tars'])} tars, {indexed_bytes / 1e9:.1f} GB): "
                    "cancelling remaining tar walks (download phase needs the rest)")
                for fu in pending:
                    fu.cancel()
                break
            if now > deadline_ts and (usable() or stop_reached()):
                log(f"index soft deadline reached with a usable index ({len(idx['tars'])} tars, "
                    f"{indexed_bytes / 1e9:.1f} GB): reserving the rest of prep for downloads")
                for fu in pending:
                    fu.cancel()
                break
            if now > deadline_ts and not usable():
                # too small to guarantee a valid subset -> keep going to the hard deadline
                if time.time() - last_tick > 120:
                    log(f"  index past soft deadline but only {len(idx['tars'])} tars: continuing to hard deadline")
                    last_tick = time.time()
            if stop_reached():
                log(f"enough archives indexed ({indexed_bytes / 1e9:.1f} GB across {len(idx['tars'])} tars): "
                    "cancelling remaining tar walks")
                for fu in pending:
                    fu.cancel()
                break
            try:
                done_fu = next(as_completed(list(pending), timeout=30), None)
            except TimeoutError:
                if time.time() - last_tick > 120:
                    log(f"  indexing... {len(pending)} tars in flight, {len(idx['tars'])} done, {indexed_bytes / 1e9:.1f} GB")
                    last_tick = time.time()
                continue
            fpath = pending.pop(done_fu)
            inflight.discard(fpath)
            try:
                path, url, mem = done_fu.result()
            except _Deadline:
                log(f"  {fpath}: walk aborted at indexing deadline (will resume next run)")
                continue
            except Exception as e:  # noqa
                log(f"  index FAILED for {fpath}: {repr(e)[:200]}")
                submit()
                continue
            n = 0
            for name, off, size in mem:
                base = os.path.basename(name)
                if not base.endswith(".zarr.zip"):
                    continue
                try:
                    sid = int(base[: -len(".zarr.zip")])
                except ValueError:
                    log("  skip unparsable member", name)
                    continue
                if str(sid) not in idx["shots"]:
                    idx["shots"][str(sid)] = [url, off, size]
                    indexed_bytes += size
                n += 1
            idx["tars"][path] = n
            log(f"  indexed {path}: {n} shots ({indexed_bytes / 1e9:.1f} GB total indexed)")
            json.dump(idx, open(P["index"], "w"))
            submit()
    missing = [f.path for f in files if f.path not in idx["tars"]]
    if missing and not stop_reached():
        log(f"WARNING: {len(missing)} tars not indexed (deadline/network?): {missing[:5]} - re-run prep to continue")
    log(f"index: {len(idx['shots'])} shots across {len(idx['tars'])} tars ({indexed_bytes / 1e9:.1f} GB)")
    return idx


def _quota_select(budget, official, avail, rng):
    """Random-priority selection of shots under a byte quota (order stays random)."""
    rng.shuffle(avail)
    used, chosen = 0, []
    for s in avail:
        sz = s[1]
        if used + sz > budget:
            continue  # keep scanning: smaller shots may still fit
        chosen.append(s[0])
        used += sz
    return chosen, used


def choose_shots(split, idx):
    from tokamark.data_split import read_data_split_csv
    csv, _, _ = split_files(split)
    tr, te, va = read_data_split_csv(csv)
    official = {"train": [int(s) for s in tr], "val": [int(s) for s in va], "test": [int(s) for s in te]}
    total = sum(len(v) for v in official.values())
    free = shutil.disk_usage(paths(split)["data"]).free
    budget = min(CFG["data_budget_gb"] * 1e9, max(0.0, free - 6e9))
    log(f"[{split}] official counts: " + ", ".join(f"{k}={len(v)}" for k, v in official.items()) + f" (total {total}); byte budget {budget / 1e9:.1f} GB")
    rng = random.Random(CFG["seed"])
    MIN_SHOTS = 5  # matches the fatal guard in cmd_prep
    sel, report = {}, {}
    for k, shots in official.items():
        avail = [(s, idx["shots"][str(s)][2]) for s in shots if str(s) in idx["shots"]]
        quota = budget * len(shots) / total
        rng.shuffle(avail)
        # guarantee >= MIN_SHOTS per split even when the byte quota is tiny: the 5
        # smallest available (cost-bounded), then fill the rest of the quota randomly.
        guard = sorted(avail, key=lambda t: t[1])[:MIN_SHOTS]
        guard_ids = {s for s, _ in guard}
        used1 = sum(sz for _, sz in guard)
        rest = [t for t in avail if t[0] not in guard_ids]
        extra, used2 = _quota_select(max(0.0, quota - used1), shots, rest, rng)
        chosen = [s for s, _ in guard] + extra
        used = used1 + used2
        sel[k] = chosen
        report[k] = dict(official=len(shots), in_archives=len(avail), selected=len(chosen), gb=round(used / 1e9, 2),
                         fraction_of_official=round(len(chosen) / max(1, len(shots)), 4))
    log(f"[{split}] selection: {json.dumps(report)}")
    return sel, report


def fetch_member(sid, meta, data_dir):
    url, off, size = meta
    final = os.path.join(data_dir, f"{sid}.zarr")
    if os.path.isdir(final):
        return sid, True, "cached"
    zpath = os.path.join(data_dir, f"{sid}.zip.part")
    tmpd = os.path.join(data_dir, f"{sid}.extract")
    try:
        with open(zpath, "wb") as fh:
            pos, CH = off, 64 * 2 ** 20
            while pos < off + size:
                end = min(pos + CH, off + size) - 1
                fh.write(ranged_get(url, pos, end))
                pos = end + 1
        if os.path.getsize(zpath) != size:
            raise RuntimeError("size mismatch")
        shutil.rmtree(tmpd, ignore_errors=True)
        with zipfile.ZipFile(zpath) as z:
            z.extractall(tmpd)
        root = tmpd
        if not (os.path.exists(os.path.join(root, "zarr.json")) or os.path.exists(os.path.join(root, ".zgroup"))):
            kids = [d for d in os.listdir(tmpd) if os.path.isdir(os.path.join(tmpd, d))]
            if len(kids) == 1 and (os.path.exists(os.path.join(tmpd, kids[0], "zarr.json")) or os.path.exists(os.path.join(tmpd, kids[0], ".zgroup"))):
                root = os.path.join(tmpd, kids[0])
            else:
                raise RuntimeError("no zarr root found in archive")
        import zarr
        zarr.open_group(root, mode="r")  # must open
        os.replace(root, final)
        return sid, True, "ok"
    except Exception as e:  # noqa
        return sid, False, repr(e)[:200]
    finally:
        shutil.rmtree(tmpd, ignore_errors=True)
        if os.path.exists(zpath):
            os.remove(zpath)


def cmd_prep(split):
    setup_imports()
    P = paths(split)
    os.makedirs(P["data"], exist_ok=True)
    os.makedirs(P["out"], exist_ok=True)
    deadline = T0 + CFG["prep_deadline_h"] * 3600
    # Indexing gets a soft sub-deadline (a fraction of the prep window) so a slow or
    # re-throttled index cannot starve downloads. But if the index is still short of the
    # size it needs when the soft deadline hits, build_index is allowed to use up to the
    # hard prep deadline instead - an under-sized index (which risks the fatal <5-shot
    # guard) is worse than a shorter download window. See build_index(soft, hard).
    index_frac = float(os.environ.get("TM_INDEX_TIME_FRAC", 0.25))
    index_soft = min(deadline, T0 + CFG["prep_deadline_h"] * 3600 * index_frac)
    idx = build_index(index_soft, hard_deadline=deadline)

    reuse = None
    if os.path.exists(P["sel"]) and not CFG["force_reselect"]:
        prev = json.load(open(P["sel"]))
        if prev.get("split") == split and min((len(v) for v in prev.get("shots", {}).values()), default=0) >= 5:
            reuse = prev
            log("reusing previous selection.json "
                f"({json.dumps({k: len(v) for k, v in reuse['shots'].items()})}); "
                "set TM_FORCE_RESELECT=1 to re-select shots")
    if reuse:
        sel, report = reuse["shots"], reuse["report"]
        for k in sel:
            report.setdefault(k, {})
            report[k]["official"] = report[k].get("official", len(sel[k]))
    else:
        sel, report = choose_shots(split, idx)

    # round-robin across splits so a deadline cut keeps the proportions; skip cached shots
    prep_splits = [s.strip() for s in os.environ.get("TM_PREP_SPLITS", "train,val,test").split(",") if s.strip()]
    order, lists = [], {k: [s for s in v if not os.path.isdir(os.path.join(P["data"], f"{s}.zarr"))]
                        for k, v in sel.items() if k in prep_splits}
    n_cached = sum(len(sel[k]) for k in prep_splits) - sum(len(lists[k]) for k in prep_splits)
    if n_cached:
        log(f"{n_cached} selected shots already on disk (cached)")
    while any(lists.values()):
        for k in prep_splits:
            if lists.get(k):
                order.append((k, lists[k].pop()))
    done, failed = set(), {}
    nworkers = int(os.environ.get("TM_FETCH_WORKERS", 4))
    attempts = {}
    inflight = set()
    with ThreadPoolExecutor(nworkers) as ex:
        it = iter(order)
        futs = {}
        def submit():
            try:
                k, s = next(it)
            except StopIteration:
                # re-queue shots that failed transiently (HTTP 429 etc.), bounded
                # retries, and never ones already in flight in another thread.
                cand = [(k, s) for (k, s) in order
                        if s not in done and s not in inflight and attempts.get(s, 0) < 3]
                if not cand:
                    return False
                k, s = cand[0]
                attempts[s] = attempts.get(s, 0) + 1
            futs[ex.submit(fetch_member, s, idx["shots"][str(s)], P["data"])] = (k, s)
            inflight.add(s)
            return True
        for _ in range(nworkers * 2):
            submit()
        while futs:
            fu = next(as_completed(list(futs)))
            k, s = futs.pop(fu)
            inflight.discard(s)
            sid, ok, msg = fu.result()
            if ok:
                done.add(sid)
                failed.pop(sid, None)
            else:
                failed[sid] = msg
            if len(done) % 25 == 0:
                log(f"  fetched {len(done)}/{len(order)} shots, failed {len(failed)}")
            if time.time() < deadline:
                submit()
        # one last pass for still-failed shots while time remains (paced = fewer 429s)
        for (k, s) in [t for t in order if t[1] not in done]:
            if time.time() >= deadline or s in done or attempts.get(s, 0) >= 3:
                continue
            attempts[s] = attempts.get(s, 0) + 1
            sid, ok, msg = fetch_member(s, idx["shots"][str(s)], P["data"])
            if ok:
                done.add(sid); failed.pop(sid, None)
            else:
                failed[sid] = msg
    final = {}
    for k, v in sel.items():
        if k in prep_splits:
            final[k] = [s for s in v if (s in done or os.path.isdir(os.path.join(P["data"], f"{s}.zarr")))]
        else:
            final[k] = v  # preserve the selected shot list for splits not being fetched
    if time.time() >= deadline and any(len(final[k]) < len(sel[k]) for k in prep_splits):
        log(f"prep deadline reached: using the {sum(len(final[k]) for k in prep_splits)} shots fetched so far "
            f"({sum(len(sel[k]) for k in prep_splits) - sum(len(final[k]) for k in prep_splits)} still missing)")
    report2 = {k: dict(report.get(k, {}), downloaded=len(final[k])) for k in final}
    meta = dict(split=split, shots=final, report=report2, failed=failed,
                config=dict(seed=CFG["seed"], task=CFG["task"], data_budget_gb=CFG["data_budget_gb"],
                            repo_id=CFG["repo_id"], git_revs=CFG["git_revs"], selected_at_min=round((time.time() - T0) / 60, 1)))
    json.dump(meta, open(P["sel"], "w"), indent=1)
    log(f"[{split}] FINAL shot counts: " + json.dumps({k: len(v) for k, v in final.items()}))
    if failed:
        log(f"  {len(failed)} shots failed and were excluded; first errors: {list(failed.items())[:3]}")
    req_splits = [k for k in prep_splits if k in final]
    if req_splits and min(len(final[k]) for k in req_splits) < 5:
        raise SystemExit(f"FATAL: fewer than 5 shots in some requested split ({req_splits}); refusing to train on a misleading subset. "
                         "Check network/index logs above, extend TM_DATA_GB / TM_PREP_DEADLINE_H, and re-run prep.")


# ======================================================================================
# Datasets / model (mirrors tokamark_baseline/run_training.py exactly, minus subset_of_shots)
# ======================================================================================
def build_data(split, which, shuffle_windows=True):
    setup_imports()
    import torch
    from torch.utils.data import DataLoader
    from tokamark.tasks import get_task_config, get_task_metadata
    from tokamark.tools.transforms.compose_transform import ComposeTransforms
    from tokamark.data import initialize_MAST_dataset, initialize_TokaMark_dataset
    from src.model_transform import ModelTransform_1, ModelTransform_2
    from src.trainer import model_collate_fn
    from utils import seed_worker

    P = paths(split)
    sel = json.load(open(P["sel"]))
    assert sel["split"] == split, "selection file belongs to another split protocol!"
    _, stats, outl = split_files(split)
    patch_stats_module(stats)
    cfgt = get_task_config(task_name=CFG["task"])
    cfgt["stride_window"] = 0.005  # same override as run_training.py for task_1-x/2-x
    meta = get_task_metadata(config_task=cfgt, verbose=False)
    tf = ComposeTransforms([ModelTransform_1(meta | cfgt), ModelTransform_2(meta | cfgt)])
    nw = CFG["dl_workers"]
    if nw < 0:
        nw = max(2, min(8, os.cpu_count() or 2))
    smgr = {"base_local_zarr_path": P["data"]}
    loaders, g = {}, torch.Generator()
    g.manual_seed(CFG["seed"])
    # Tensor IPC must not go through /dev/shm (Docker default is 64 MB; a 512-window
    # batch deadlocks the workers silently). file_system uses /tmp instead.
    torch.multiprocessing.set_sharing_strategy("file_system")
    for w in which:
        mast = initialize_MAST_dataset(config_task=cfgt, shots_list=sel["shots"][w], local_flag=True, use_std_scaling=True,
                                       stats_metadata_file_path=stats, use_nan_filling=False, remove_outliers=True,
                                       outlier_metadata_file=outl, remove_bad_efit_rating=True,
                                       store_manager_settings=smgr, verbose=False)
        kw = dict(test_mode=True)
        if w == "test":
            kw["shuffle_windows"] = False
        ds = initialize_TokaMark_dataset(dataset=mast, task_metadata=meta, config_metadata=cfgt, custom_transform=tf, **kw)
        if ds is None:
            raise RuntimeError(f"{w} dataset empty after official filtering (outliers/bad EFIT)")
        # TokaMarkDataset is an IterableDataset (no __getitem__), so probe via the
        # iterator: time how long one window takes to materialise. A real failure here
        # (empty dataset, bad zarr) is fatal, not a harmless warning.
        t_probe = time.time()
        first = next(iter(ds), None)
        dt = time.time() - t_probe
        if first is None:
            raise RuntimeError(f"{w} dataset yielded no windows after official filtering "
                               f"(outliers/bad EFIT / empty shots) - cannot train on it")
        est = dt * CFG["batch_size"] / max(1, min(max(nw, 1), CFG["batch_size"]))
        log(f"{w} dataset: {len(sel['shots'][w])} shots ({type(ds).__name__}, streaming windows); "
            f"first window took {dt:.2f}s "
            f"(=> first batch of {CFG['batch_size']} will take roughly {est / 60:.1f} min)")
        extra = dict(drop_last=True) if w == "train" else {}
        if nw > 0:
            extra["persistent_workers"] = True
        loaders[w] = DataLoader(ds, batch_size=CFG["batch_size"], num_workers=nw, collate_fn=model_collate_fn,
                                worker_init_fn=seed_worker, generator=g, pin_memory=torch.cuda.is_available(),
                                prefetch_factor=2 if nw > 0 else None, **extra)
        log(f"{w} loader ready: {len(sel['shots'][w])} shots, batch_size={CFG['batch_size']}, workers={nw}")
    return loaders, meta | cfgt


# ======================================================================================
# MODULE 2: FAILURE-AWARE ARCHITECTURE & PINN LOSS (MASK DROP + FLUX-LOOP CONSISTENCY)
# ======================================================================================
FLUX_LOOP_COORDS = (
    (2, 13), (2, 22), (2, 32), (2, 42), (2, 51),
    (60, 6), (60, 13), (60, 19), (60, 26), (60, 32),
    (60, 38), (60, 45), (60, 51), (60, 58), (60, 62)
)


def flux_loop_pinn_loss(pred_psi, true_psi, mask_fl=None):
    """Enforce boundary flux consistency at the 15 wall flux loop locations."""
    loss = 0.0
    for r_idx, z_idx in FLUX_LOOP_COORDS:
        diff_sq = (pred_psi[:, 0, z_idx, r_idx] - true_psi[:, 0, z_idx, r_idx]) ** 2
        if mask_fl is not None:
            diff_sq = diff_sq * mask_fl
        loss = loss + diff_sq.mean()
    return loss / len(FLUX_LOOP_COORDS)


def apply_structured_failure(x, p_fail=0.5):
    """Structured failure augmentation: drops channels, correlated groups, or temporal gaps."""
    import torch
    B = x[0].shape[0]
    dev = x[0].device
    dtype = x[0].dtype
    mask = torch.ones((B, 8), device=dev, dtype=dtype)
    x_aug = [t.clone() for t in x]
    groups = [[0], [1, 2, 3], [4], [5, 6], [7]]
    for b in range(B):
        if torch.rand(1).item() < p_fail:
            r = torch.rand(1).item()
            if r < 0.40:
                k = int(torch.randint(1, 4, (1,)).item())
                chans = torch.randperm(8)[:k].tolist()
                for c in chans:
                    x_aug[c][b] = 0.0
                    mask[b, c] = 0.0
            elif r < 0.70:
                grp = groups[int(torch.randint(0, len(groups), (1,)).item())]
                for c in grp:
                    x_aug[c][b] = 0.0
                    mask[b, c] = 0.0
            else:
                k = int(torch.randint(1, 3, (1,)).item())
                chans = torch.randperm(8)[:k].tolist()
                for c in chans:
                    L = x_aug[c].shape[-1]
                    half = max(1, L // 2)
                    start = int(torch.randint(0, max(1, L - half + 1), (1,)).item())
                    x_aug[c][b, :, :, start:start + half] = 0.0
                    mask[b, c] = 0.0
    return x_aug, mask


def _resolve_mask(model, args, kwargs):
    """Split forward args into (inputs, exogenous, health mask (B, n_in)).

    A missing mask is inferred from the inputs: a branch that is exactly zero counts as dead."""
    import torch
    n_in, n_exo = len(model.input_branches), len(model.exogenous_branches)
    inputs, exogenous = args[:n_in], args[n_in:n_in + n_exo]
    mask = kwargs.get("mask", None)
    if mask is None and len(args) > n_in + n_exo:
        mask = args[n_in + n_exo]
    B, dev, dtype = inputs[0].shape[0], inputs[0].device, inputs[0].dtype
    if mask is None:
        mask = torch.ones((B, n_in), device=dev, dtype=dtype)
        for i, x_chan in enumerate(inputs):
            is_zero = (x_chan.abs().sum(dim=tuple(range(1, x_chan.dim()))) == 0.0)
            mask[:, i] = torch.where(is_zero, 0.0, 1.0)
    elif mask.dim() == 1:
        mask = mask.unsqueeze(0).expand(B, -1)
    return inputs, exogenous, mask


def make_failure_aware_model(loader, meta):
    """Module 2 network for the chosen backbone (TM_MODEL = mlp | lstm).

    Both keep the upstream one-CNN-encoder-per-sensor layout. Each encoder output is gated by its sensor's health
    flag, and the flag vector is also fed to the backbone (MLP bottleneck / LSTM input) so it knows what is missing."""
    setup_imports()
    import torch
    import torch.nn as nn
    from torch.utils.checkpoint import checkpoint
    is_lstm = CFG["model"].lower().strip() == "lstm"
    if is_lstm:
        from src.multi_conv_lstm_model import LSTM_v3 as Base, bb_factor
    else:
        from src.multi_conv_mlp_model import MultiConv_MLP as Base, bb_factor
    from tokamark.tools.utils import get_device

    device = get_device()
    input_shapes, exogenous_shapes, output_shapes = [], [], []
    for l, first_window in enumerate(loader.dataset):
        try:
            input_shapes = [arr.shape for arr in first_window["input"]]
            exogenous_shapes = [arr.shape for arr in first_window["exogenous"]]
            output_shapes = [arr.shape for arr in first_window["y"]]
            break
        except Exception:
            continue

    class MultiConv_FailureAware_MLP(Base):
        def __init__(self, in_shapes, exo_shapes, out_shapes, dict_meta, D=16):
            super().__init__(in_shapes, exo_shapes, out_shapes, dict_meta, D=D)
            self.n_in = len(self.input_branches)
            in_features = self.D * bb_factor * self.n_in * self.W_in + self.n_in
            self.encoder_mlp = nn.Sequential(
                nn.Linear(in_features, 2 * self.D * bb_factor),
                nn.ReLU(),
                nn.Linear(2 * self.D * bb_factor, self.D * bb_factor),
            )

        def forward(self, *args, **kwargs):
            n_in = self.n_in
            n_exo = len(self.exogenous_branches)
            inputs = args[:n_in]
            exogenous = args[n_in:n_in + n_exo]
            mask = kwargs.get("mask", None)
            if mask is None and len(args) > n_in + n_exo:
                mask = args[n_in + n_exo]
            B = inputs[0].shape[0]
            dev = inputs[0].device
            dtype = inputs[0].dtype
            if mask is None:
                mask = torch.ones((B, n_in), device=dev, dtype=dtype)
                for i, x_chan in enumerate(inputs):
                    is_zero = (x_chan.abs().sum(dim=tuple(range(1, x_chan.dim()))) == 0.0)
                    mask[:, i] = torch.where(is_zero, 0.0, 1.0)
            elif mask.dim() == 1:
                mask = mask.unsqueeze(0).expand(B, -1)

            input_branch_outputs = []
            for i, (branch, x) in enumerate(zip(self.input_branches, inputs)):
                out = checkpoint(self._run_cnn_encoder, branch, x, use_reentrant=False)
                m_i = mask[:, i].view(-1, 1, 1)
                out = out * m_i
                input_branch_outputs.append(out)

            input_seq = torch.cat(input_branch_outputs, dim=2)
            enc_in = input_seq.reshape(B, -1)
            enc_in = torch.cat([enc_in, mask], dim=1)
            context = self.encoder_mlp(enc_in)

            if len(self.exogenous_branches) > 0 and len(exogenous) > 0:
                exo_branch_outputs = [checkpoint(self._run_cnn_encoder, b, x, use_reentrant=False)
                                      for b, x in zip(self.exogenous_branches, exogenous)]
                exo_seq = torch.cat(exo_branch_outputs, dim=2)
                exo_flat = exo_seq.reshape(B, -1)
                decoder_in = torch.cat([context, exo_flat], dim=1)
            else:
                decoder_in = context

            dec_flat = self.decoder_mlp(decoder_in)
            dec_out = dec_flat.unsqueeze(1).repeat(1, self.W_out, 1)

            outputs = []
            for branch, goal_shape in zip(self.output_branches, self.output_shapes):
                out = checkpoint(self._run_cnn_decoder, branch, goal_shape, dec_out, use_reentrant=False)
                outputs.append(out)
            return outputs

    class MultiConv_FailureAware_LSTM(Base):
        def __init__(self, in_shapes, exo_shapes, out_shapes, dict_meta, D=16):
            super().__init__(in_shapes, exo_shapes, out_shapes, dict_meta, D=D)
            self.n_in = len(self.input_branches)
            old = self.encoder_lstm  # same LSTM, with room for the health flags appended to every timestep
            self.encoder_lstm = nn.LSTM(input_size=old.input_size + self.n_in, hidden_size=old.hidden_size,
                                        num_layers=old.num_layers, batch_first=True, dropout=old.dropout)

        def forward(self, *args, **kwargs):
            inputs, exogenous, mask = _resolve_mask(self, args, kwargs)

            input_branch_outputs = []
            for i, (branch, x) in enumerate(zip(self.input_branches, inputs)):
                out = checkpoint(self._run_cnn_encoder, branch, x, use_reentrant=False)
                input_branch_outputs.append(out * mask[:, i].view(-1, 1, 1))  # mask feature gating h_i * m_i
            input_seq = torch.cat(input_branch_outputs, dim=2)
            input_seq = torch.cat([input_seq, mask.unsqueeze(1).expand(-1, input_seq.shape[1], -1)], dim=2)

            enc_out, (h, c) = self.encoder_lstm(input_seq)
            decoder_input = self.start_token.to(enc_out.device).repeat(enc_out.size(0), self.W_out, 1)
            if len(self.exogenous_branches) > 0 and len(exogenous) > 0:
                exo_branch_outputs = [checkpoint(self._run_cnn_encoder, b, x, use_reentrant=False)
                                      for b, x in zip(self.exogenous_branches, exogenous)]
                decoder_input = torch.cat([decoder_input, torch.cat(exo_branch_outputs, dim=2)], dim=2)
            dec_out, _ = self.decoder_lstm(decoder_input, (h, c))

            return [checkpoint(self._run_cnn_decoder, branch, goal_shape, dec_out, use_reentrant=False)
                    for branch, goal_shape in zip(self.output_branches, self.output_shapes)]

    FailureAware = MultiConv_FailureAware_LSTM if is_lstm else MultiConv_FailureAware_MLP
    return FailureAware(in_shapes=input_shapes, exo_shapes=exogenous_shapes, out_shapes=output_shapes,
                        dict_meta=meta, D=16).to(device)


def make_model(loader, meta):
    if CFG["failure_aware"]:
        return make_failure_aware_model(loader, meta)
    model_type = os.environ.get("TM_MODEL", "mlp").lower().strip()
    if model_type == "lstm":
        from src.multi_conv_lstm_model import create_lstm_architecture
        return create_lstm_architecture(dataloader_=loader, dict_metadata=meta, verbose=False)
    else:
        from src.multi_conv_mlp_model import create_cnn_architecture
        return create_cnn_architecture(dataloader_=loader, dict_metadata=meta, verbose=False)


# ======================================================================================
# TRAIN: resumable, deadline-aware, optional DataParallel (benchmarked)
# ======================================================================================
def atomic_save(obj, path):
    import torch
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)


def pick_parallel(model, loader, dev, criterion, move):
    """Return (module_to_call, label). Benchmarks 1 GPU vs DataParallel on one cached batch."""
    import torch, torch.nn as nn
    n = torch.cuda.device_count()
    if n < 2 or CFG["use_dp"] == "0":
        return model, f"single GPU ({n} visible)"
    if CFG["use_dp"] == "1":
        return nn.DataParallel(model), f"DataParallel x{n} (forced)"
    try:
        batch = first_batch_timed(loader, tag="benchmark batch")
        x, y = move(batch, dev)
        def bench(m):
            opt = torch.optim.SGD(m.parameters(), lr=0.0)  # lr 0: benchmark does not change weights
            m.train()
            for i in range(14):
                if i == 4:
                    torch.cuda.synchronize(); t = time.time()
                opt.zero_grad()
                if CFG["failure_aware"]:
                    c_mask = torch.ones((x[0].shape[0], 8), device=dev, dtype=x[0].dtype)
                    criterion(m(*x, c_mask), y).backward()
                else:
                    criterion(m(*x), y).backward()
                opt.step()
            torch.cuda.synchronize()
            return (time.time() - t) / 10
        sd = {k: v.clone() for k, v in model.state_dict().items()}
        t1 = bench(model)
        dp = nn.DataParallel(model)
        t2 = bench(dp)
        model.load_state_dict(sd)
        log(f"GPU benchmark (compute only, same batch): 1 GPU {t1 * 1e3:.0f} ms/step, DataParallel {t2 * 1e3:.0f} ms/step")
        if t2 < t1 / 1.15:
            return dp, f"DataParallel x{n} (benchmark {t1 / t2:.2f}x faster)"
        return model, "single GPU (DataParallel not clearly faster)"
    except Exception as e:  # noqa
        log("parallel benchmark failed, using single GPU:", repr(e)[:200])
        return model, "single GPU (benchmark error)"


def cmd_train(split):
    setup_imports()
    import torch
    from src.trainer import MultiOutputMSELoss, move_batch_to_device
    from utils import set_seed
    set_seed(CFG["seed"])
    P = paths(split)
    os.makedirs(P["model_dir"], exist_ok=True)
    sel = json.load(open(P["sel"]))
    run_config = dict(task=CFG["task"], model=CFG["model"], split=split, seed=CFG["seed"], batch_size=CFG["batch_size"],
                      lr=CFG["lr"], weight_decay=CFG["weight_decay"], max_steps=CFG["max_steps"],
                      validate_every=CFG["validate_every"], patience=CFG["patience"],
                      train_deadline_h=CFG["train_deadline_h"], use_dp=CFG["use_dp"],
                      failure_aware=CFG["failure_aware"], fail_aug_p=CFG["fail_aug_p"],
                      pinn_lambda=CFG["pinn_lambda"],
                      git_revs=CFG["git_revs"],
                      shots={k: len(v) for k, v in sel["shots"].items()},
                      report=sel["report"])
    json.dump(run_config, open(os.path.join(P["out"], "run_config.json"), "w"), indent=1)
    log(f"run_config: {json.dumps(run_config)}")
    dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    log(f"device={dev}, gpus={torch.cuda.device_count()}, cpus={os.cpu_count()}")
    phase = {"p": "building dataloaders", "step": 0}

    def _heartbeat():
        while True:
            time.sleep(60)
            log(f"  heartbeat: {phase['p']} (step {phase['step']})")

    threading.Thread(target=_heartbeat, daemon=True).start()
    loaders, meta = build_data(split, ["train", "val"])
    phase["p"] = "building model"
    t_model = time.time()
    log("building model (shape inference walks the dataset for one sample)...")
    model = make_model(loaders["train"], meta).to(dev)
    log(f"model built in {time.time() - t_model:.1f}s, "
        f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params")
    crit = MultiOutputMSELoss()

    # ---- fail-fast smoke check (a few seconds): shapes, NaNs, forward/backward ----
    phase["p"] = "first batch (smoke)"
    b = first_batch_timed(loaders["train"], tag="first train batch")
    sid, wid, xs, ys = b
    x, y = move_batch_to_device(b, dev)
    if CFG["failure_aware"]:
        clean_mask = torch.ones((x[0].shape[0], 8), device=dev, dtype=x[0].dtype)
        out = model(*x, clean_mask)
    else:
        out = model(*x)
    log("smoke: input shapes", [tuple(t.shape) for t in x], "| target shapes", [tuple(t.shape) for t in y],
        "| output shapes", [tuple(o.shape) for o in out])
    assert all(o.shape == t.shape for o, t in zip(out, y)), "output/target shape mismatch"
    assert all(torch.isfinite(t).all() for t in x), "non-finite inputs after collate"
    l = crit(out, y); l.backward()
    assert torch.isfinite(l), "non-finite initial loss"
    log(f"smoke OK: initial loss {l.item():.4f}")
    model.zero_grad(set_to_none=True)

    opt = torch.optim.Adam(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3, min_lr=1e-6)
    st = dict(step=0, best=float("inf"), no_improve=0, epoch=0, oom=0,
              hist=dict(step=[], train_loss=[], val_loss=[], lr=[], t=[], mse=[], pinn=[]))
    last = os.path.join(P["model_dir"], "last.pt")
    if os.path.exists(last) and not CFG["train_from_scratch"]:
        ck = torch.load(last, map_location=dev, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); st = ck["state"]
        if "sched" in ck and ck["sched"] is not None:
            sched.load_state_dict(ck["sched"])
        st.setdefault("oom", 0)
        st["hist"].setdefault("t", [])
        st["hist"].setdefault("lr", [])
        st["hist"].setdefault("mse", [])
        st["hist"].setdefault("pinn", [])
        log(f"RESUMED from step {st['step']} (best val {st['best']:.5f})")
    elif CFG["train_from_scratch"] and os.path.exists(last):
        log("TM_TRAIN_FROM_SCRATCH=1: ignoring existing checkpoint, training from step 0")
    runner, label = pick_parallel(model, loaders["train"], dev, crit, move_batch_to_device) if st["step"] == 0 else (model, "single GPU (resumed)")
    log("parallel mode:", label)

    @torch.no_grad()
    def validate():
        phase["p"] = "validation"
        runner.eval(); tot = n = 0
        for vb in loaders["val"]:
            if vb is None: continue
            vx, vy = move_batch_to_device(vb, dev)
            bs = vy[0].shape[0]
            if CFG["failure_aware"]:
                clean_mask = torch.ones((bs, 8), device=dev, dtype=vx[0].dtype)
                lv = crit(runner(*vx, clean_mask), vy)
            else:
                lv = crit(runner(*vx), vy)
            tot += lv.item() * bs; n += bs
        phase["p"] = "training loop"
        if n == 0:
            raise RuntimeError("validation loader produced zero samples - check selection.json / val shots")
        return tot / n

    def save_last():
        sched_st = sched.state_dict() if sched is not None else None
        atomic_save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched_st, state=st), last)
        atomic_save(st["hist"], os.path.join(P["model_dir"], "training_history.pt"))
        json.dump(st["hist"], open(os.path.join(P["model_dir"], "training_history.json"), "w"))

    deadline = T0 + CFG["train_deadline_h"] * 3600
    phase["p"] = "training loop"
    it, run_loss, errs, bad, oom, t_last_ck, t_tr = iter(loaders["train"]), [], 0, 0, 0, time.time(), time.time()
    stop_reason = "max_steps"
    run_parts = []  # (mse, pinn) per step, for the loss-component plot
    while st["step"] < CFG["max_steps"]:
        if time.time() > deadline:
            stop_reason = "time deadline"; break
        phase["p"] = f"waiting for batch (step {st['step']})"
        try:
            batch = next(it)
        except StopIteration:
            it = iter(loaders["train"]); st["epoch"] += 1; run_loss = []; run_parts = []; log(f"epoch {st['epoch']} done"); continue
        except Exception as e:  # noqa  (e.g. a corrupt shot inside a worker)
            errs += 1; log(f"data error #{errs}: {repr(e)[:200]}")
            if errs > 20: raise
            it = iter(loaders["train"]); continue
        if batch is None: continue
        phase["p"] = f"train step {st['step'] + 1}"
        try:
            runner.train()
            x, y = move_batch_to_device(batch, dev)
            opt.zero_grad(set_to_none=True)
            if CFG["failure_aware"]:
                x_train, mask_train = apply_structured_failure(x, p_fail=CFG["fail_aug_p"])
                out = runner(*x_train, mask_train)
                loss_mse = crit(out, y)
                if CFG["pinn_lambda"] > 0:
                    loss_fl = flux_loop_pinn_loss(out[0], y[0], mask_fl=mask_train[:, 0])
                    loss = loss_mse + CFG["pinn_lambda"] * loss_fl
                    run_parts.append((loss_mse.item(), loss_fl.item()))
                else:
                    loss = loss_mse
            else:
                loss = crit(runner(*x), y)
            if not torch.isfinite(loss):
                bad += 1; log(f"non-finite loss, skipping (#{bad})")
                if bad > 200: raise RuntimeError("too many non-finite losses")
                continue
            bad = 0
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
        except torch.cuda.OutOfMemoryError:
            oom += 1; st["oom"] = oom
            torch.cuda.empty_cache()
            log(f"CUDA OOM at step {st['step']} (#{oom}) - batch skipped; current batch_size={CFG['batch_size']}")
            if oom >= 3:
                raise RuntimeError(
                    f"CUDA out of memory {oom}x with batch_size={CFG['batch_size']} on this GPU. "
                    "Re-run the train cell with a smaller TM_BATCH in the config cell (e.g. 256 or 128); "
                    f"training will resume from {last} without losing progress.")
            continue
        st["step"] += 1; run_loss.append(loss.item()); phase["step"] = st["step"]
        if st["step"] <= 10 or st["step"] % 20 == 0:
            el = time.time() - t_tr; t_tr = time.time()
            log(f"step {st['step']} loss {sum(run_loss) / len(run_loss):.4f} "
                f"({(CFG['batch_size'] if st['step'] <= 10 else 20 * CFG['batch_size']) / el:.0f} samples/s)")
        if st["step"] == 1:
            save_last()  # keep progress even if the session dies early
        if st["step"] % CFG["validate_every"] == 0:
            v = validate(); h = st["hist"]
            cur_lr = opt.param_groups[0]["lr"]
            for k, val in zip(("step", "train_loss", "val_loss", "lr", "t"), (st["step"], sum(run_loss) / len(run_loss), v, cur_lr, time.time() - T0)):
                h[k].append(val)
            if run_parts:
                h["mse"].append(sum(a for a, _ in run_parts) / len(run_parts))
                h["pinn"].append(sum(b for _, b in run_parts) / len(run_parts))
            sched.step(v)
            if v < st["best"]:
                st["best"], st["no_improve"] = v, 0
                atomic_save(model.state_dict(), os.path.join(P["model_dir"], "best_model.pt"))
            else:
                st["no_improve"] += 1
            log(f"VAL step {st['step']} val_loss {v:.5f} best {st['best']:.5f} no_improve {st['no_improve']}/{CFG['patience']}")
            save_last(); t_last_ck = time.time()
            if st["no_improve"] >= CFG["patience"]:
                stop_reason = "early stopping (official patience rule)"; break
        elif time.time() - t_last_ck > CFG["ckpt_every_s"]:
            save_last(); t_last_ck = time.time()
    v = validate()
    if v < st["best"] or not os.path.exists(os.path.join(P["model_dir"], "best_model.pt")):
        st["best"] = v; atomic_save(model.state_dict(), os.path.join(P["model_dir"], "best_model.pt"))
    save_last()
    summary = dict(stop_reason=stop_reason, steps=st["step"], epochs=st["epoch"], best_val=st["best"],
                   parallel=label, cuda_oom_events=st.get("oom", 0), data_errors=errs,
                   duration_min=round((time.time() - T0) / 60, 1),
                   model_dir=P["model_dir"], **{k: run_config[k] for k in ("task", "split", "seed", "batch_size", "lr",
                                                                           "weight_decay", "validate_every", "patience",
                                                                           "git_revs", "shots", "report")})
    json.dump(summary, open(os.path.join(P["out"], "train_summary.json"), "w"), indent=1)
    log(f"TRAIN DONE ({stop_reason}) at step {st['step']}, best val {st['best']:.5f}; summary -> train_summary.json")


# ======================================================================================
# EVAL: official evaluator on test shots, un-standardized NRMSE
# ======================================================================================
def cmd_eval(split):
    setup_imports()
    import torch
    from tokamark.evaluator import WindowMetricsAccumulator, compute_metrics
    from src.evaluator import cnn_unstd_evaluation_per_shot
    P = paths(split)
    os.makedirs(P["results"], exist_ok=True)
    if not os.path.exists(os.path.join(P["model_dir"], "best_model.pt")):
        raise SystemExit(f"FATAL: {P['model_dir']}best_model.pt not found - run train first")
    loaders, meta = build_data(split, ["test"])
    model = make_model(loaders["test"], meta)
    acc = WindowMetricsAccumulator(CFG["task"])
    cnn_unstd_evaluation_per_shot(test_dataloader=loaders["test"], config_task=meta, cnn_model=model,
                                  output_dir=P["model_dir"], accumulator=acc)
    compute_metrics(task=CFG["task"], output_dir=P["results"], window_metrics_accumulator=acc,
                    save_windows_metrics=True, save_task_metrics=True)
    sel = json.load(open(P["sel"]))
    log(f"[{split}] EVAL DONE. Results in {P['results']}")
    log(f"[{split}] shot counts used: {json.dumps(sel['report'])}")
    for f in sorted(os.listdir(P["results"])):
        if f.endswith((".json", ".yaml", ".yml")):
            print(f"--- {f}\n" + open(os.path.join(P["results"], f)).read()[:3000])
    for root, _, files in os.walk(P["results"]):
        for f in sorted(files):
            if f.endswith(".csv"):
                print(f"--- {os.path.join(root, f)}")
                with open(os.path.join(root, f)) as fh:
                    for i, line in enumerate(fh):
                        if i > 12: print("  ..."); break
                        print("  " + line.rstrip())


# ======================================================================================
# PLOT: training dashboard, reconstruction samples, LCFS overlay, error diagnostics, per-shot error
# (never blocks core results)
# ======================================================================================
_DR_CM, _DZ_CM = 180.0 / 64, 300.0 / 64          # MAST 65x65 grid: cm per pixel (R, Z)
_PSI_EXTENT = [0.2, 2.0, -1.5, 1.5]               # R [m], Z [m]; array rows = Z, columns = R


def _run_title():
    kind = "failure-aware + PINN" if CFG["failure_aware"] else "baseline"
    return f"TokaMark {CFG['task']} CNN+{CFG['model'].upper()} ({kind})"


def _lcfs_segments(field, level):
    """Contour vertices (col, row) of `field` at `level`; short speckle segments are dropped."""
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots()
    segs = [s for s in ax.contour(field, levels=[level]).allsegs[0] if len(s) >= 8]
    plt.close(fig)
    return segs


def _boundary_offset_cm(true, pred, frac=0.35):
    """Mean symmetric distance (cm) between the true and predicted LCFS, both cut at the TRUE field's flux level."""
    import numpy as np
    level = true.min() + frac * (true.max() - true.min())
    st, sp = _lcfs_segments(true, level), _lcfs_segments(pred, level)
    if not st or not sp:
        return None
    a = np.concatenate(st) * [_DR_CM, _DZ_CM]
    b = np.concatenate(sp) * [_DR_CM, _DZ_CM]
    d = np.sqrt(((a[:, None, :] - b[None, :, :]) ** 2).sum(-1))
    return float(0.5 * (d.min(1).mean() + d.min(0).mean()))


def _fig_loss_dashboard(h, path, title):
    import numpy as np
    import matplotlib.pyplot as plt
    steps, tr, va = np.array(h["step"]), np.array(h["train_loss"]), np.array(h["val_loss"])
    best_i = int(np.argmin(va))
    fig, ax = plt.subplots(2, 2, figsize=(12, 7.5))
    a = ax[0, 0]
    a.plot(steps, tr, color="#4C78A8", lw=1.8, label="train (running mean)")
    a.plot(steps, va, color="#E4572E", lw=1.8, label="validation")
    a.axvline(steps[best_i], color="grey", ls="--", lw=1)
    a.scatter([steps[best_i]], [va[best_i]], color="#E4572E", zorder=5, s=45)
    a.annotate(f"best val {va[best_i]:.4f}\nstep {int(steps[best_i])}", (steps[best_i], va[best_i]),
               textcoords="offset points", xytext=(12, -42), fontsize=8,
               arrowprops=dict(arrowstyle="->", color="grey", lw=0.8))
    a.set_title("Train vs validation loss"); a.set_xlabel("step"); a.set_ylabel("MultiOutputMSELoss (standardised)")
    a.legend(); a.grid(alpha=0.3)
    a = ax[0, 1]
    a.fill_between(steps, 0, va - tr, color="#9467BD", alpha=0.35)
    a.plot(steps, va - tr, color="#9467BD", lw=1.5)
    a.axvline(steps[best_i], color="grey", ls="--", lw=1)
    a.set_title("Generalisation gap (val - train)"); a.set_xlabel("step"); a.set_ylabel("loss gap"); a.grid(alpha=0.3)
    a = ax[1, 0]
    if h.get("lr"):
        a.semilogy(steps[:len(h["lr"])], h["lr"], color="#2CA02C", drawstyle="steps-post", lw=1.8)
    a.set_title("Learning rate (ReduceLROnPlateau)"); a.set_xlabel("step"); a.set_ylabel("lr"); a.grid(alpha=0.3, which="both")
    a = ax[1, 1]
    if h.get("mse") and h.get("pinn") and len(h["mse"]) == len(steps):
        a.plot(steps, h["mse"], color="#4C78A8", lw=1.8, label="MSE term")
        a.set_ylabel("MSE term", color="#4C78A8"); a.set_xlabel("step")
        b = a.twinx()
        b.plot(steps, h["pinn"], color="#F58518", lw=1.8, label="flux-loop PINN term (unweighted)")
        b.set_ylabel("PINN term", color="#F58518")
        a.set_title("Loss components (train, running mean)")
    elif h.get("t"):
        a.plot(steps[:len(h["t"])], np.array(h["t"]) / 60, color="#17BECF", lw=1.8)
        a.set_title("Wall-clock time"); a.set_xlabel("step"); a.set_ylabel("minutes since session start")
    a.grid(alpha=0.3)
    fig.suptitle(title, fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    fig.savefig(path, dpi=130)
    plt.close(fig)


def _fig_samples(true, pred, labels, path, title):
    """true/pred: (n, H, W) in Wb. One row per window: true | pred | |error|."""
    import numpy as np
    import matplotlib.pyplot as plt
    n = len(true)
    fig, axes = plt.subplots(n, 3, figsize=(12, 3.6 * n), squeeze=False)
    for i in range(n):
        err = np.abs(pred[i] - true[i])
        lo, hi = min(true[i].min(), pred[i].min()), max(true[i].max(), pred[i].max())
        for j, (img, ttl, cm, kw) in enumerate((
                (true[i], "True psi", "viridis", dict(vmin=lo, vmax=hi)),
                (pred[i], "Predicted psi", "viridis", dict(vmin=lo, vmax=hi)),
                (err, f"|Error|  (MAE {err.mean():.4f} Wb, max {err.max():.4f})", "magma", {}))):
            im = axes[i][j].imshow(img, cmap=cm, origin="lower", extent=_PSI_EXTENT, aspect="auto", **kw)
            axes[i][j].set_title(f"{ttl}\n{labels[i]}" if j == 0 else ttl, fontsize=9)
            axes[i][j].set_xlabel("R [m]", fontsize=8); axes[i][j].set_ylabel("Z [m]", fontsize=8)
            fig.colorbar(im, ax=axes[i][j], fraction=0.046)
    fig.suptitle(title, fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    fig.savefig(path, dpi=130)
    plt.close(fig)


def _fig_boundary(true, pred, labels, path, title):
    import numpy as np
    import matplotlib.pyplot as plt
    n = len(true)
    fig, axes = plt.subplots(1, n, figsize=(5.2 * n, 5.2), squeeze=False)
    for i in range(n):
        a = axes[0][i]
        level = true[i].min() + 0.35 * (true[i].max() - true[i].min())
        a.imshow(true[i], cmap="Greys", origin="lower", extent=_PSI_EXTENT, aspect="auto", alpha=0.55)
        x = np.linspace(_PSI_EXTENT[0], _PSI_EXTENT[1], true[i].shape[1])
        z = np.linspace(_PSI_EXTENT[2], _PSI_EXTENT[3], true[i].shape[0])
        a.contour(x, z, true[i], levels=[level], colors="#1F77B4", linewidths=2.2)
        a.contour(x, z, pred[i], levels=[level], colors="#D62728", linewidths=1.8, linestyles="--")
        off = _boundary_offset_cm(true[i], pred[i])
        a.set_title(f"{labels[i]}\nLCFS offset: " + (f"{off:.2f} cm" if off is not None else "n/a"), fontsize=9)
        a.set_xlabel("R [m]"); a.set_ylabel("Z [m]")
    axes[0][0].plot([], [], color="#1F77B4", lw=2.2, label="true LCFS")
    axes[0][0].plot([], [], color="#D62728", lw=1.8, ls="--", label="predicted LCFS")
    axes[0][0].legend(loc="upper right", fontsize=8)
    fig.suptitle(title + " - last closed flux surface (35% level of the true field)", fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    fig.savefig(path, dpi=130)
    plt.close(fig)


def _fig_error_diagnostics(err_map, win_mae, sc_t, sc_p, path, title):
    import numpy as np
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(2, 2, figsize=(12, 9))
    a = ax[0, 0]
    im = a.imshow(err_map, cmap="magma", origin="lower", extent=_PSI_EXTENT, aspect="auto")
    fig.colorbar(im, ax=a, fraction=0.046, label="mean |error| [Wb]")
    fl = np.array(FLUX_LOOP_COORDS)
    a.scatter(_PSI_EXTENT[0] + fl[:, 0] / 64 * 1.8, _PSI_EXTENT[2] + fl[:, 1] / 64 * 3.0, s=22, facecolors="none",
              edgecolors="cyan", linewidths=1, label="15 flux loops (PINN)")
    a.set_title("Mean absolute error over all test windows"); a.set_xlabel("R [m]"); a.set_ylabel("Z [m]")
    a.legend(loc="upper right", fontsize=8)
    a = ax[0, 1]
    a.hist(win_mae, bins=40, color="#4C78A8", alpha=0.9)
    a.axvline(np.mean(win_mae), color="#E4572E", lw=1.8, label=f"mean {np.mean(win_mae):.4f}")
    a.axvline(np.median(win_mae), color="k", ls="--", lw=1.5, label=f"median {np.median(win_mae):.4f}")
    a.set_title(f"Per-window MAE ({len(win_mae)} windows)"); a.set_xlabel("MAE [Wb]"); a.set_ylabel("windows")
    a.legend(); a.grid(alpha=0.3)
    a = ax[1, 0]
    hb = a.hexbin(sc_t, sc_p, gridsize=60, bins="log", cmap="viridis", mincnt=1)
    lim = [min(sc_t.min(), sc_p.min()), max(sc_t.max(), sc_p.max())]
    a.plot(lim, lim, color="white", lw=1.2, ls="--")
    r2 = 1.0 - np.sum((sc_t - sc_p) ** 2) / max(np.sum((sc_t - sc_t.mean()) ** 2), 1e-12)
    a.set_title(f"Predicted vs true psi (R2 = {r2:.4f})"); a.set_xlabel("true psi [Wb]"); a.set_ylabel("predicted psi [Wb]")
    fig.colorbar(hb, ax=a, fraction=0.046, label="log10 count")
    a = ax[1, 1]
    fl_err = np.array([err_map[z, r] for r, z in FLUX_LOOP_COORDS])
    a.bar(range(len(fl_err)), fl_err, color="#F58518")
    a.axhline(err_map.mean(), color="k", ls="--", lw=1.3, label=f"grid-wide mean {err_map.mean():.4f}")
    a.set_title("Error at the 15 flux-loop pixels (PINN anchor points)")
    a.set_xlabel("flux loop #"); a.set_ylabel("mean |error| [Wb]"); a.legend(); a.grid(alpha=0.3, axis="y")
    fig.suptitle(title, fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    fig.savefig(path, dpi=130)
    plt.close(fig)


def _fig_per_shot(csv_path, path, title):
    import csv as _csv
    import numpy as np
    import matplotlib.pyplot as plt
    rows = list(_csv.DictReader(open(csv_path)))
    by_shot = {}
    for r in rows:
        by_shot.setdefault(r["shot_id"], []).append((int(r["window_index"]), float(r["MAE"])))
    order = sorted(by_shot, key=lambda s: np.mean([m for _, m in by_shot[s]]))
    means = [float(np.mean([m for _, m in by_shot[s]])) for s in order]
    overall = float(np.mean([float(r["MAE"]) for r in rows]))
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    a = ax[0]
    a.bar(range(len(order)), means, color=["#E4572E" if m > overall else "#4C78A8" for m in means])
    a.axhline(overall, color="k", ls="--", lw=1.3, label=f"all windows {overall:.4f}")
    a.set_xticks(range(len(order))); a.set_xticklabels(order, rotation=90, fontsize=6)
    a.set_title(f"Mean MAE per test shot ({len(order)} shots, red = above average)")
    a.set_ylabel("MAE [Wb]"); a.legend(); a.grid(alpha=0.3, axis="y")
    a = ax[1]
    grid = np.linspace(0, 1, 21)
    curves = []
    for s in by_shot:
        w = sorted(by_shot[s])
        if len(w) < 5:
            continue
        pos = np.linspace(0, 1, len(w))
        curves.append(np.interp(grid, pos, [m for _, m in w]))
    if curves:
        c = np.array(curves)
        a.fill_between(grid, np.percentile(c, 10, axis=0), np.percentile(c, 90, axis=0), color="#4C78A8", alpha=0.25,
                       label="10-90th percentile over shots")
        a.plot(grid, np.median(c, axis=0), color="#4C78A8", lw=2, label="median shot")
    a.set_title("Error along the discharge (normalised time)"); a.set_xlabel("normalised time in shot"); a.set_ylabel("window MAE [Wb]")
    a.legend(); a.grid(alpha=0.3)
    fig.suptitle(title, fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    fig.savefig(path, dpi=130)
    plt.close(fig)


def cmd_plot(split):
    setup_imports()
    import numpy as np
    import torch
    import matplotlib
    matplotlib.use("Agg")
    P = paths(split)
    os.makedirs(P["figures"], exist_ok=True)
    title = f"{_run_title()} - {split} split, seed {CFG['seed']}"

    # (a) training dashboard - from the saved history
    try:
        h = torch.load(os.path.join(P["model_dir"], "training_history.pt"), map_location="cpu", weights_only=False)
        _fig_loss_dashboard(h, os.path.join(P["figures"], "loss_curves.png"), title)
        log(f"saved loss_curves.png ({len(h['step'])} validation points)")
    except Exception as e:  # noqa
        log("loss-curve plot failed:", repr(e)[:200])

    # (b)-(d) run the best model over the test windows once and feed the reconstruction figures
    try:
        from src.trainer import move_batch_to_device
        dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        loaders, meta = build_data(split, ["test"])
        model = make_model(loaders["test"], meta).to(dev)
        model.load_state_dict(torch.load(os.path.join(P["model_dir"], "best_model.pt"), map_location=dev, weights_only=False))
        model.eval()
        ps = meta["output"]["equilibrium-psi"]
        mu, sd = float(ps["mean"]), float(ps["std"])
        err_sum, n_win, win_mae, sc_t, sc_p, first = None, 0, [], [], [], None
        rng = np.random.default_rng(0)
        for bi, batch in enumerate(loaders["test"]):
            if batch is None:
                continue
            if bi >= 8:
                break
            x, y = move_batch_to_device(batch, dev)
            with torch.no_grad():
                yp = model(*x)
            t = y[0][:, 0].float().cpu().numpy() * sd + mu
            p = yp[0][:, 0].float().cpu().numpy() * sd + mu
            e = np.abs(p - t)
            err_sum = e.sum(0) if err_sum is None else err_sum + e.sum(0)
            n_win += len(e)
            win_mae.extend(e.mean((1, 2)).tolist())
            pick = rng.choice(t.size, size=min(20000, t.size), replace=False)
            sc_t.append(t.ravel()[pick]); sc_p.append(p.ravel()[pick])
            if first is None:
                first = (t, p, [int(s) for s in batch[0]])
        if n_win == 0:
            raise RuntimeError("no test windows")
        win_mae = np.array(win_mae)
        t, p, sids = first
        wm = np.abs(p - t).mean((1, 2))
        pick = [int(np.argmin(wm)), int(np.argsort(wm)[len(wm) // 2]), int(np.argmax(wm))]
        labels = [f"{nm}: shot {sids[i]}, MAE {wm[i]:.4f} Wb" for nm, i in zip(("best", "median", "worst"), pick)]
        _fig_samples(t[pick], p[pick], labels, os.path.join(P["figures"], "psi_reconstruction_samples.png"), title)
        log("saved psi_reconstruction_samples.png")
        _fig_boundary(t[pick], p[pick], labels, os.path.join(P["figures"], "boundary_overlay.png"), title)
        log("saved boundary_overlay.png")
        _fig_error_diagnostics(err_sum / n_win, win_mae, np.concatenate(sc_t), np.concatenate(sc_p),
                               os.path.join(P["figures"], "test_error_diagnostics.png"), title)
        log(f"saved test_error_diagnostics.png ({n_win} test windows)")
    except Exception as e:  # noqa
        log("reconstruction figures failed (core results unaffected):", repr(e)[:300])

    # (e) per-shot error from the official evaluator's window metrics
    try:
        wm_csv = os.path.join(P["results"], CFG["task"], "windows_metrics.csv")
        _fig_per_shot(wm_csv, os.path.join(P["figures"], "per_shot_error.png"), title)
        log("saved per_shot_error.png")
    except Exception as e:  # noqa
        log("per-shot figure failed (run eval first):", repr(e)[:200])
    log(f"figures in {P['figures']}")


# ======================================================================================
# SUMMARY: one human-readable recap of the whole run
# ======================================================================================
def cmd_summary(split):
    setup_imports()
    import csv as _csv
    P = paths(split)
    lines = [f"# TokaMark {CFG['task']} baseline run - {split} split", ""]
    def add_json(fname, title):
        p = os.path.join(P["out"], fname)
        if os.path.exists(p):
            lines.append(f"## {title}")
            lines.append("```json")
            lines.append(json.dumps(json.load(open(p)), indent=1))
            lines.append("```")
        else:
            lines.append(f"## {title}\n\nMISSING: {p}")
        lines.append("")
    add_json("selection.json", "Data subset & split (official membership, budget-limited selection)")
    add_json("run_config.json", "Run configuration")
    add_json("train_summary.json", "Training summary")
    lines.append("## Evaluation (official evaluator, un-standardised NRMSE)")
    tm = os.path.join(P["results"], CFG["task"], "task_metrics.csv")
    if os.path.exists(tm):
        rows = list(_csv.DictReader(open(tm)))
        for r in rows:
            name = r["feature_name"]
            lines.append(f"- **{name}**: NRMSE={float(r['NRMSE_mean']):.4f} (+/-{float(r['NRMSE_std_pop']):.4f}), "
                         f"RMSE={float(r['RMSE_mean']):.5f}, MAE={float(r['MAE_mean']):.5f}, n_shots={r['n_shots']}")
    else:
        lines.append(f"MISSING: {tm} (eval not run?)")
    lines.append("")
    lines.append("## Artifacts")
    for label, p in (("selection", P["sel"]), ("run config", os.path.join(P["out"], "run_config.json")),
                     ("train summary", os.path.join(P["out"], "train_summary.json")),
                     ("best model", os.path.join(P["model_dir"], "best_model.pt")),
                     ("resume checkpoint", os.path.join(P["model_dir"], "last.pt")),
                     ("history", os.path.join(P["model_dir"], "training_history.pt")),
                     ("results dir", P["results"]), ("figures dir", P["figures"])):
        lines.append(f"- {label}: `{p}` {'OK' if os.path.exists(p) else '**MISSING**'}")
    text = "\n".join(lines)
    with open(os.path.join(P["out"], "summary.md"), "w") as fh:
        fh.write(text)
    print(text)
    log(f"summary -> {os.path.join(P['out'], 'summary.md')}")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=["prep", "train", "eval", "plot", "summary"])
    ap.add_argument("--split", default="random", choices=["random", "temporal"])
    a = ap.parse_args()
    {"prep": cmd_prep, "train": cmd_train, "eval": cmd_eval, "plot": cmd_plot, "summary": cmd_summary}[a.cmd](a.split)


In [ ]:
%%writefile /kaggle/working/robustness.py
#!/usr/bin/env python
"""TokaMark task_1-3 sensor-robustness harness: corrupt | eval-robust | report.

Stress-tests a *trained, frozen* task_1-3 model under the six sensor-failure
scenarios of Gupta (arXiv 2607.11915), on the TEST split only, with optional
imputation. Reuses the exact pipeline helpers (build_data / make_model) and the
official un-standardise + hierarchical-metric protocol so the CLEAN baseline
reproduces the eval-time NRMSE before any corruption is trusted.

Commands:
  python robustness.py corrupt     --split random     # build corrupted test loaders + manifest
  python robustness.py eval-robust --split random     # run the sweep, save metrics
  python robustness.py report      --split random     # figures + robustness.md

All settings come from TM_* env vars (same CFG contract as pipeline.py). Robustness
knobs:
  TM_ROB_SCENARIOS  comma list; default: random_dropout,channel_ablation,category_abl,
                    temporal_gap,group_failure,disruption_proximate  (or 'all')
  TM_ROB_IMPUTE     zero,mean,ffill (comma list); default: zero,mean,ffill
  TM_ROB_SEED       RNG seed for corruption (default TM_SEED)
  TM_ROB_CLEAN_ONLY 1 -> only the clean pass (sanity check), skip corruption
  TM_ROB_BATCH      forward batch size (default TM_BATCH)
  TM_ROB_PHYS_STRIDE  every Nth test window also gets the Module 1 physics metrics (boundary / axis / X-point
                    error in cm, wall-alarm rates), stored as "PHYS" in robust_metrics.json (default 100; 0 = off)
"""
import os, sys, json, time, math, argparse
import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pipeline as P
from pipeline import CFG, T0, log, setup_imports, build_data, make_model

# Module 1 (physics metrics in cm) is optional here: it lives in ../modules/ in the repo, next to this file on Kaggle (the notebook writes it there).
sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "modules"))
try:
    from physics_evaluation import evaluate_window_physics, aggregate_physics_metrics
except ImportError:  # e.g. the Kaggle notebook, which only writes pipeline.py and this file
    evaluate_window_physics = aggregate_physics_metrics = None

TASK = CFG["task"]

SIGNALS = [
    "magnetics/flux_loop_flux",
    "magnetics/b_field_pol_probe_ccbv_field",
    "magnetics/b_field_pol_probe_obr_field",
    "magnetics/b_field_pol_probe_obv_field",
    "magnetics/b_field_tor_probe_saddle_voltage",
    "pf_active/coil_current",
    "pf_active/solenoid_current",
    "summary/ip",
]
SIGNAL_SHORT = ["flux_loop", "ccbv", "obr", "obv", "saddle_v", "coil_i", "solenoid_i", "ip"]

CATEGORY_GROUPS = {
    "flux_loop": [0],
    "pickup_fields": [1, 2, 3],
    "saddle_voltage": [4],
    "active_coils": [5],
    "solenoid": [6],
    "plasma_current": [7],
}
CORRELATED_GROUPS = {
    "magnetic_probes": [1, 2, 3],
    "power_supply": [5, 6],
    "magnetics_all": [0, 1, 2, 3, 4],
}

SCENARIOS = ["random_dropout", "channel_ablation", "category_abl",
             "temporal_gap", "group_failure", "disruption_proximate"]
SEVERITIES = {
    "random_dropout": [0.10, 0.25, 0.50],
    "channel_ablation": [1, 3, 6],
    "category_abl": [1],
    "temporal_gap": [0.20, 0.40, 0.60],
    "group_failure": [1],
    "disruption_proximate": [0.10, 0.25, 0.50],
}
IMPUTE_MODES = ("zero", "mean", "ffill")


def envf(k, d):
    return type(d)(os.environ.get(k, d))


def rob_cfg():
    scen = os.environ.get("TM_ROB_SCENARIOS", "all")
    if scen.strip() == "all":
        scen = SCENARIOS
    else:
        scen = [s.strip() for s in scen.split(",") if s.strip()]
    imp = [m.strip() for m in os.environ.get("TM_ROB_IMPUTE", "zero,mean,ffill").split(",") if m.strip()]
    bad = [s for s in scen if s not in SCENARIOS]
    bad += [m for m in imp if m not in IMPUTE_MODES]
    if bad:
        raise SystemExit(f"FATAL: unknown robustness options: {bad}")
    return dict(
        scenarios=scen,
        imputes=imp,
        seed=envf("TM_ROB_SEED", CFG["seed"]),
        clean_only=os.environ.get("TM_ROB_CLEAN_ONLY", "0") == "1",
        batch=envf("TM_ROB_BATCH", CFG["batch_size"]),
        phys_stride=envf("TM_ROB_PHYS_STRIDE", 100),
    )


def rob_paths(split):
    out = os.path.join(CFG["work"], "out", split)
    rdir = os.path.join(out, "robustness")
    return dict(
        dir=rdir,
        manifest=os.path.join(rdir, "robust_manifest.json"),
        metrics=os.path.join(rdir, "robust_metrics.json"),
        windows=os.path.join(rdir, "robust_windows.csv"),
        figures=os.path.join(rdir, "figures"),
        report=os.path.join(rdir, "robustness.md"),
        model_dir=os.path.join(out, "model") + "/",
    )


# ======================================================================================
# CORRUPTION PRIMITIVES  (operate on a per-channel torch tensor x: (B, W, C, L))
# ======================================================================================
def _as_np(t):
    return t.detach().cpu().numpy() if hasattr(t, "detach") else np.asarray(t)


def _mask_random_dropout(arr, sev, rng):
    """Zero a fraction `sev` of the *finite, non-zero* values (Gupta's definition).

    Returns (masked_array, hole_mask) where hole_mask marks exactly the values we zeroed.
    """
    m = np.isfinite(arr) & (arr != 0)
    idx = np.flatnonzero(m.ravel())
    out = arr.copy()
    holes = np.zeros_like(arr, dtype=bool)
    if idx.size == 0:
        return out, holes
    k = int(round(sev * idx.size))
    drop = rng.choice(idx, size=min(k, idx.size), replace=False)
    out.ravel()[drop] = 0.0
    holes.ravel()[drop] = True
    return out, holes


def _mask_temporal_gap(arr, frac, rng, where="random"):
    """Zero a contiguous fraction `frac` of the time axis (last axis)."""
    out = arr.copy()
    holes = np.zeros_like(arr, dtype=bool)
    L = arr.shape[-1]
    g = max(1, int(round(frac * L)))
    if where == "front":
        s = 0
    elif where == "end":
        s = L - g
    else:
        s = int(rng.integers(0, max(1, L - g + 1)))
    out[..., s:s + g] = 0.0
    holes[..., s:s + g] = True
    return out, holes


def _zero_channels(arr, chans):
    """Zero whole channels; return (masked, holes) with holes marking the zeroed area."""
    out = arr.copy()
    holes = np.zeros_like(arr, dtype=bool)
    for c in chans:
        out[..., c, :] = 0.0
        holes[..., c, :] = True
    return out, holes


def _impute(masked, holes, mode, channel_mean=0.0):
    """Fill ONLY the positions flagged in `holes`. mode in {zero,mean,ffill}.

    'zero'   -> leave the zeros (TokaMark default; no filling).
    'mean'   -> put `channel_mean` (TRAIN-split signal mean) in every hole.
    'ffill'  -> carry the last valid (non-hole) value forward along time, per channel.
    """
    if mode == "zero":
        return masked
    out = np.array(masked, dtype=masked.dtype, copy=True)
    if not holes.any():
        return out
    if mode == "mean":
        out[holes] = float(channel_mean)
        return out
    if mode == "ffill":
        ow = out.reshape(-1, out.shape[-2], out.shape[-1])
        hw = holes.reshape(-1, holes.shape[-2], holes.shape[-1])
        for i in range(ow.shape[0]):
            for c in range(ow.shape[1]):
                row, hole = ow[i, c], hw[i, c]
                last = 0.0
                for t in range(row.shape[0]):
                    if hole[t]:
                        row[t] = last
                    else:
                        last = row[t]
        return out
    raise ValueError(mode)


# ======================================================================================
# SCENARIO APPLICATION  (returns a new inputs list, corruption bookkeeping dict)
# ======================================================================================
def apply_scenario(inputs, scenario, sev, impute, train_means, rng):
    """inputs: list of 8 torch tensors (B, W, C, L). Returns (corrupted_inputs, info).

    Corruption is applied on CPU in numpy, then re-wrapped to torch tensors matching
    each input's original dtype. Imputation 'mean' uses `train_means` (per signal, a
    scalar computed from the TRAIN split) so nothing about the test split leaks.
    """
    import torch
    info = dict(scenario=scenario, severity=sev, impute=impute, details={}, dead_channels=[])
    # channel-set scenarios are deterministic per task (draw once, not per channel)
    ablate_chans = None
    if scenario == "channel_ablation":
        n = min(int(sev), len(SIGNALS))
        ablate_chans = sorted(rng.choice(len(SIGNALS), size=n, replace=False).tolist())
        info["details"]["channels_zeroed"] = [SIGNAL_SHORT[c] for c in ablate_chans]
        info["dead_channels"] = ablate_chans  # whole sensors lost: the failure-aware model is told via its health mask
    out = []
    for i, t in enumerate(inputs):
        arr = _as_np(t).astype(np.float64)
        masked, holes = arr, np.zeros_like(arr, dtype=bool)
        if scenario == "random_dropout":
            masked, holes = _mask_random_dropout(arr, float(sev), rng)
        elif scenario == "channel_ablation":
            if i in ablate_chans:
                masked, holes = _zero_channels(arr, [0])
        elif scenario == "temporal_gap":
            masked, holes = _mask_temporal_gap(arr, float(sev), rng, where="random")
        elif scenario == "disruption_proximate":
            masked, holes = _mask_temporal_gap(arr, float(sev), rng, where="end")
        else:
            raise ValueError(scenario)
        masked = _impute(masked, holes, impute, channel_mean=train_means.get(i, 0.0))
        out.append(torch.as_tensor(masked, dtype=getattr(t, "dtype", None)))
    return out, info


def apply_group(inputs, scenario, sev, impute, train_means, rng, group_key, group_chans):
    """Like apply_scenario but zeroes a fixed set of channels (category / group runs)."""
    import torch
    group_chans = [c for c in group_chans if 0 <= c < len(inputs)]
    info = dict(scenario=scenario, severity=sev, impute=impute, dead_channels=list(group_chans),
                details={"group": group_key, "channels": [SIGNAL_SHORT[c] for c in group_chans]})
    out = []
    for i, t in enumerate(inputs):
        arr = _as_np(t).astype(np.float64)
        if i in group_chans:
            masked, holes = _zero_channels(arr, [0])
        else:
            masked, holes = arr, np.zeros_like(arr, dtype=bool)
        masked = _impute(masked, holes, impute, channel_mean=train_means.get(i, 0.0))
        out.append(torch.as_tensor(masked, dtype=getattr(t, "dtype", None)))
    return out, info


# ======================================================================================
# METRICS  (reuse the official hierarchy: per-window -> per-shot -> across shots)
# ======================================================================================
def _psi_mean_std(meta):
    m = meta["output"]["equilibrium-psi"]
    return float(m["mean"]), float(m["std"])


def window_metrics(y_true, y_pred):
    """y_true/y_pred: (N, 65, 65) numpy, real units. Returns per-window RMSE, MAE arrays."""
    diff = y_pred - y_true
    rmse = np.sqrt(np.nanmean(diff ** 2, axis=(1, 2)))
    mae = np.nanmean(np.abs(diff), axis=(1, 2))
    return rmse, mae


def aggregate(shot_ids, rmse, mae, std):
    """Per-window -> per-shot mean -> across-shot mean/std; plus window-level stats.

    Mirrors tokamark.evaluator: RMSE per window, then averaged per shot, then NRMSE =
    shot_RMSE / output_std, then mean/std across shots (population sd, ddof=0).
    """
    shot_ids = np.asarray(shot_ids, dtype=np.int64)
    rmse = np.asarray(rmse, dtype=np.float64)
    mae = np.asarray(mae, dtype=np.float64)
    uniq = np.unique(shot_ids)
    shot_rmse, shot_mae = [], []
    for s in uniq:
        m = shot_ids == s
        # Official TokaMark: shot RMSE = sqrt(mean(window_RMSE ** 2))
        shot_rmse.append(float(np.sqrt(np.nanmean(rmse[m] ** 2))))
        shot_mae.append(float(np.nanmean(mae[m])))
    shot_rmse = np.asarray(shot_rmse)
    shot_mae = np.asarray(shot_mae)
    return dict(
        n_shots=int(len(uniq)),
        n_windows=int(len(rmse)),
        NRMSE=float(np.nanmean(shot_rmse / std)),
        NRMSE_std=float(np.nanstd(shot_rmse / std, ddof=0)),
        NMAE=float(np.nanmean(shot_mae / std)),
        NMAE_std=float(np.nanstd(shot_mae / std, ddof=0)),
        RMSE=float(np.nanmean(shot_rmse)),
        RMSE_std=float(np.nanstd(shot_rmse, ddof=0)),
        MAE=float(np.nanmean(shot_mae)),
        MAE_std=float(np.nanstd(shot_mae, ddof=0)),
        window_RMSE_mean=float(np.nanmean(rmse)),
        window_RMSE_median=float(np.nanmedian(rmse)),
        window_RMSE_p90=float(np.nanpercentile(rmse, 90)),
        window_RMSE_max=float(np.nanmax(rmse)),
    )


# ======================================================================================
# INFERENCE PASS
# ======================================================================================
def load_test_loader_and_model(split, batch=None):
    setup_imports()
    import torch
    loaders, meta = build_data(split, ["test"])
    if batch is not None:
        # rebuild the loader with the requested batch size (build_data uses CFG batch)
        from torch.utils.data import DataLoader
        from src.trainer import model_collate_fn
        from utils import seed_worker
        ds = loaders["test"].dataset
        nw = CFG["dl_workers"]
        if nw < 0:
            nw = max(2, min(8, os.cpu_count() or 2))
        g = torch.Generator(); g.manual_seed(CFG["seed"])
        loaders["test"] = DataLoader(ds, batch_size=batch, num_workers=nw, collate_fn=model_collate_fn,
                                     worker_init_fn=seed_worker, generator=g,
                                     pin_memory=torch.cuda.is_available(),
                                     prefetch_factor=2 if nw > 0 else None)
    model = make_model(loaders["test"], meta)
    dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model.load_state_dict(torch.load(os.path.join(rob_paths(split)["model_dir"], "best_model.pt"),
                                     map_location=dev, weights_only=False))
    model.to(dev).eval()
    n_gpu = torch.cuda.device_count()
    if n_gpu > 1 and CFG["use_dp"] != "0":  # inference is pure forward passes: split every batch across all GPUs
        model = torch.nn.DataParallel(model)
    log(f"inference on {max(n_gpu, 1) if CFG['use_dp'] != '0' else 1} GPU(s) ({n_gpu} visible)")
    return loaders["test"], model, meta, dev


def train_channel_means(split):
    """Per-signal mean computed on the TRAIN subset (for mean-imputation, no leakage)."""
    setup_imports()
    try:
        P_ = P.paths(split)
        sel = json.load(open(P_["sel"]))
        train_shots = [s for s in sel.get("shots", {}).get("train", [])
                       if os.path.isdir(os.path.join(P_["data"], f"{s}.zarr"))]
        if len(train_shots) >= 5:
            loaders, _ = build_data(split, ["train"])
            return _train_means_pass(loaders["train"])
        else:
            log(f"only {len(train_shots)} train shots on disk; using baseline 0.0 for mean imputation")
            return {i: 0.0 for i in range(len(SIGNALS))}
    except Exception as e:  # noqa
        log("train-channel-mean pass failed, falling back to zeros:", repr(e)[:200])
        return {i: 0.0 for i in range(len(SIGNALS))}


def _train_means_pass(loader, max_batches=40):
    sums = None; cnts = None
    seen = 0
    for batch in loader:
        if batch is None:
            continue
        x = batch[2]
        if sums is None:
            sums = [0.0] * len(x); cnts = [0] * len(x)
        for i, t in enumerate(x):
            a = _as_np(t).astype(np.float64)
            a = a[np.isfinite(a) & (a != 0)]
            sums[i] += float(a.sum()); cnts[i] += int(a.size)
        seen += 1
        if seen >= max_batches:
            break
    means = {}
    for i in range(len(sums)):
        means[i] = (sums[i] / cnts[i]) if cnts[i] > 0 else 0.0
    log(f"train channel means (first {seen} batches): " +
        ", ".join(f"{SIGNAL_SHORT[i]}={means[i]:.3g}" for i in sorted(means)))
    return means


def run_pass(loader, model, dev, mu, sd, corrupt_fn=None, accumulate_spatial=False, return_viz=False, phys_stride=0):
    """Forward the whole test loader; return aggregated metrics + per-window rows.

    corrupt_fn: callable(x_list) -> (x_corrupted_list, info) applied per batch, or None
                for the clean pass.
    phys_stride: every Nth window (same windows in every pass) also gets the Module 1 physics metrics.
    A failure-aware model is given a health mask: 0 for every sensor the scenario removed entirely
    (channel / category / group loss), 1 for partial damage (dropout, gaps), which it must handle unaided.
    """
    import torch
    phys_rows, n_seen = [], 0
    sid_all, wid_all, rmse_all, mae_all = [], [], [], []
    spatial_sum = None; spatial_cnt = None
    viz_data = None
    n_batches = 0
    with torch.no_grad():
        for batch in loader:
            if batch is None:
                continue
            shot_id, window_id, x, y = batch
            x = [t.to(dev, dtype=torch.float32) for t in x]
            xin, info = x, {}
            if corrupt_fn is not None:
                xin_cpu, info = corrupt_fn(x)
                xin = [t.to(dev, dtype=torch.float32) for t in xin_cpu]
            if CFG["failure_aware"]:
                health = torch.ones((xin[0].shape[0], len(xin)), device=dev)
                health[:, info.get("dead_channels", [])] = 0.0
                y_pred = model(*xin, health)
            else:
                y_pred = model(*xin)
            if not isinstance(y_pred, (list, tuple)):
                y_pred = [y_pred]
            yt = y[0].detach().cpu().squeeze(1).numpy().astype(np.float64) * sd + mu
            yp = y_pred[0].detach().cpu().squeeze(1).numpy().astype(np.float64) * sd + mu
            yt = yt.reshape(yt.shape[0], -1, 65, 65)
            yp = yp.reshape(yp.shape[0], -1, 65, 65)
            # flatten any extra window axis into the sample axis
            n = yt.shape[0] * yt.shape[1]
            yt2 = yt.reshape(n, 65, 65)
            yp2 = yp.reshape(n, 65, 65)
            if return_viz and viz_data is None and n > 0:
                sids_raw = np.asarray(shot_id).reshape(-1)
                viz_data = dict(yt=yt2[0].tolist(), yp=yp2[0].tolist(),
                                shot_id=int(sids_raw[0]) if len(sids_raw) > 0 else 0)
            if phys_stride and evaluate_window_physics is not None:
                phys_rows += [evaluate_window_physics(yt2[j], yp2[j]) for j in range(n) if (n_seen + j) % phys_stride == 0]
            n_seen += n
            rmse, mae = window_metrics(yt2, yp2)
            rmse = rmse.reshape(yt.shape[0], yt.shape[1]).mean(axis=1)
            mae = mae.reshape(yt.shape[0], yt.shape[1]).mean(axis=1)
            sid_all.append(np.asarray(shot_id).reshape(-1))
            wid_all.append(np.asarray(window_id).reshape(-1))
            rmse_all.append(rmse); mae_all.append(mae)
            if accumulate_spatial:
                err = np.abs(yp2 - yt2)
                s = np.nansum(err, axis=0); c = np.isfinite(err).sum(axis=0)
                spatial_sum = s if spatial_sum is None else spatial_sum + s
                spatial_cnt = c if spatial_cnt is None else spatial_cnt + c
            n_batches += 1
    sid = np.concatenate(sid_all) if len(sid_all) > 0 else np.array([])
    rmse = np.concatenate(rmse_all) if len(rmse_all) > 0 else np.array([])
    mae = np.concatenate(mae_all) if len(mae_all) > 0 else np.array([])
    m = aggregate(sid, rmse, mae, sd)
    m["n_batches"] = n_batches
    if phys_rows:
        m["PHYS"] = aggregate_physics_metrics(phys_rows)
    if accumulate_spatial and spatial_sum is not None and spatial_cnt is not None:
        m["spatial_abs_err"] = (spatial_sum / np.maximum(spatial_cnt, 1)).tolist()
    if return_viz and viz_data is not None:
        m["viz"] = viz_data
    m["_windows"] = dict(shot_id=sid.tolist(), rmse=rmse.tolist(), mae=mae.tolist())
    return m


def build_sweep_tasks(rc):
    """Expand the sweep into a list of task dicts (id, callable-corruption spec)."""
    tasks = []
    for scen in rc["scenarios"]:
        for sev in SEVERITIES[scen]:
            for imp in rc["imputes"]:
                if scen in ("category_abl", "group_failure"):
                    groups = CATEGORY_GROUPS if scen == "category_abl" else CORRELATED_GROUPS
                    for gname, gchans in groups.items():
                        tasks.append(dict(id=f"{scen}|{gname}|{imp}", scenario=scen,
                                          severity=sev, impute=imp, group=gname, chans=gchans))
                else:
                    tasks.append(dict(id=f"{scen}|{sev}|{imp}", scenario=scen,
                                      severity=sev, impute=imp, group=None, chans=None))
    return tasks


def make_corrupt_fn(task, rng_seed, train_means):
    """Return a callable(x_list)->(x_corrupt,info) with a fresh RNG per task (seeded)."""
    rng = np.random.default_rng(rng_seed)
    if task["scenario"] in ("category_abl", "group_failure"):
        def fn(x):
            return apply_group(x, task["scenario"], task["severity"], task["impute"],
                               train_means, rng, task["group"], task["chans"])
    else:
        def fn(x):
            return apply_scenario(x, task["scenario"], task["severity"], task["impute"],
                                  train_means, rng)
    return fn


# ======================================================================================
# COMMANDS
# ======================================================================================
def cmd_corrupt(split):
    """Build + persist the sweep manifest (and verify the clean baseline gate)."""
    rc = rob_cfg()
    P_ = rob_paths(split); os.makedirs(P_["dir"], exist_ok=True)
    tasks = build_sweep_tasks(rc)
    json.dump(dict(config=rc, tasks=tasks, signals=SIGNALS, categories=CATEGORY_GROUPS,
                   correlated_groups=CORRELATED_GROUPS, severities=SEVERITIES),
              open(P_["manifest"], "w"), indent=1)
    log(f"corruption manifest: {len(tasks)} tasks -> {P_['manifest']}")
    log("sweep tasks:\n  " + "\n  ".join(t["id"] for t in tasks))


def cmd_eval_robust(split):
    rc = rob_cfg()
    P_ = rob_paths(split); os.makedirs(P_["dir"], exist_ok=True)
    loader, model, meta, dev = load_test_loader_and_model(split, batch=rc["batch"])
    mu, sd = _psi_mean_std(meta)

    # ---- CLEAN GATE: must reproduce the eval-time NRMSE before corruption is trusted
    log("CLEAN pass (sensor-healthy baseline)...")
    clean = run_pass(loader, model, dev, mu, sd, accumulate_spatial=True, return_viz=True, phys_stride=rc["phys_stride"])
    viz_samples = dict(clean=clean.pop("viz", None), tasks={})
    log(f"CLEAN: NRMSE={clean['NRMSE']:.4f} (±{clean['NRMSE_std']:.4f}) NMAE={clean['NMAE']:.4f} "
        f"RMSE={clean['RMSE']:.5f} MAE={clean['MAE']:.5f} n_shots={clean['n_shots']} n_windows={clean['n_windows']}")
    targets = None
    mp = os.path.join(P_["model_dir"], "..", "results", "task_1-3", "task_metrics.csv")
    if os.path.exists(mp):
        import csv as _csv
        with open(mp) as fh:
            for row in _csv.DictReader(fh):
                feat = row.get("feature_name", "")
                if feat in ("equilibrium-psi", "equilibrium_psi", "task_1-3") or "psi" in feat:
                    val = row.get("NRMSE_mean") or row.get("NRMSE")
                    if val:
                        try:
                            targets = float(val)
                        except Exception:
                            pass
                        break
    if targets is not None:
        d = abs(clean["NRMSE"] - targets)
        ok = d <= max(0.02, 0.15 * targets)
        log(f"clean-gate: harness NRMSE={clean['NRMSE']:.4f} vs eval CSV={targets:.4f} "
            f"(Δ={d:.4f}) -> {'PASS' if ok else 'FAIL (trust nothing downstream)'}")
        if not ok:
            log("WARNING: clean baseline mismatch; corruption numbers below are suspect")

    result = dict(clean=clean, corrupted={}, config=rc, psi_mean=mu, psi_std=sd, viz_samples=viz_samples)
    if rc["clean_only"]:
        clean.pop("_windows", None)
        json.dump(result, open(P_["metrics"], "w"), indent=1)
        log("TM_ROB_CLEAN_ONLY=1 -> wrote clean-only metrics, skipping corruption")
        return

    log("building TRAIN channel means for mean-imputation...")
    train_means = train_channel_means(split)

    VIZ_TASKS = {"random_dropout|0.5|zero", "random_dropout|0.5|ffill",
                 "category_abl|plasma_current|zero", "category_abl|plasma_current|ffill"}
    tasks = build_sweep_tasks(rc)
    log(f"corruption sweep: {len(tasks)} tasks on the TEST split...")
    rows = []
    for ti, task in enumerate(tasks):
        cf = make_corrupt_fn(task, rc["seed"] * 1000003 + ti, train_means)
        need_viz = task["id"] in VIZ_TASKS
        log(f"[{ti + 1}/{len(tasks)}] {task['id']} ...")
        try:
            m = run_pass(loader, model, dev, mu, sd, corrupt_fn=cf, return_viz=need_viz, phys_stride=rc["phys_stride"])
        except Exception as e:  # noqa
            log(f"  task {task['id']} FAILED: {repr(e)[:200]}")
            continue
        if need_viz and "viz" in m:
            viz_samples["tasks"][task["id"]] = m.pop("viz", None)
        m.pop("_windows", None)
        m["id"] = task["id"]
        rs = clean["NRMSE"] / m["NRMSE"] if m["NRMSE"] > 0 else float("nan")
        m["RS_vs_clean"] = float(rs)
        result["corrupted"][task["id"]] = m
        log(f"  {task['id']}: NRMSE={m['NRMSE']:.4f} (+{100 * (m['NRMSE'] / clean['NRMSE'] - 1):.1f}%) "
            f"NMAE={m['NMAE']:.4f} RS={rs:.3f}")
        rows.append(dict(id=task["id"], scenario=task["scenario"], severity=task["severity"],
                         impute=task["impute"], group=task["group"], **{k: m[k] for k in (
                             "NRMSE", "NMAE", "RMSE", "MAE", "RS_vs_clean", "n_shots", "n_windows")}))
        json.dump(result, open(P_["metrics"], "w"), indent=1)  # incremental save

    clean.pop("_windows", None)
    json.dump(result, open(P_["metrics"], "w"), indent=1)
    log(f"robustness metrics written -> {P_['metrics']} ({len(rows)} corrupted tasks)")


def cmd_report(split):
    rc = rob_cfg()
    P_ = rob_paths(split)
    if not os.path.exists(P_["metrics"]):
        raise SystemExit(f"FATAL: {P_['metrics']} not found - run eval-robust first")
    res = json.load(open(P_["metrics"]))
    os.makedirs(P_["figures"], exist_ok=True)
    clean = res["clean"]; corrupted = res["corrupted"]

    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    # ---- RS aggregation: worst-case NRMSE per scenario (zero-fill), Gupta-style
    def rs_table():
        worst = {}
        for tid, m in corrupted.items():
            scen, _, imp = tid.split("|")
            if imp != "zero":
                continue
            worst[scen] = max(worst.get(scen, 0.0), m["NRMSE"])
        rs = {scen: (clean["NRMSE"] / v if v else float("nan")) for scen, v in worst.items()}
        return worst, rs

    worst, rs = rs_table()

    # ---- figure 1: degradation curves NRMSE vs severity, per scenario (zero-fill)
    try:
        fig, ax = plt.subplots(figsize=(8, 5))
        for scen in SCENARIOS:
            pts = []
            for tid, m in corrupted.items():
                parts = tid.split("|")
                if parts[0] == scen and parts[-1] == "zero" and len(parts) == 3:
                    try:
                        pts.append((float(parts[1]), m["NRMSE"]))
                    except ValueError:  # named severity (e.g. group_failure|flux_loop): not a point on a numeric curve
                        pass
            if pts:
                pts.sort()
                ax.plot([p[0] for p in pts], [p[1] for p in pts], marker="o", label=scen)
        ax.axhline(clean["NRMSE"], ls="--", c="k", lw=1, label="clean")
        ax.set_xlabel("severity"); ax.set_ylabel("NRMSE (test, un-standardised)")
        ax.set_title(f"Robustness degradation - {TASK} ({split})")
        ax.legend(fontsize=8); ax.grid(alpha=0.3); fig.tight_layout()
        fig.savefig(os.path.join(P_["figures"], "degradation_curves.png"), dpi=120)
        plt.close(fig)
        log("saved degradation_curves.png")
    except Exception as e:  # noqa
        log("degradation curve failed:", repr(e)[:200])

    # ---- figure 2: per-signal importance (category_abl + channel_ablation n=1), zero-fill
    try:
        labels, vals = [], []
        for tid, m in corrupted.items():
            p = tid.split("|")
            if p[0] == "category_abl" and p[-1] == "zero":
                labels.append(p[1]); vals.append(100 * (m["NRMSE"] / clean["NRMSE"] - 1))
        order = np.argsort(vals)[::-1]
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar([labels[i] for i in order], [vals[i] for i in order], color="steelblue")
        ax.set_ylabel("% NRMSE increase vs clean"); ax.grid(alpha=0.3, axis="y")
        ax.set_title(f"Diagnostic importance (zero each category) - {TASK}")
        fig.tight_layout(); fig.savefig(os.path.join(P_["figures"], "signal_importance.png"), dpi=120)
        plt.close(fig)
        log("saved signal_importance.png")
    except Exception as e:  # noqa
        log("importance bar failed:", repr(e)[:200])

    # ---- figure 3: spatial error map (clean)
    try:
        sm = np.array(clean.get("spatial_abs_err", []))
        if sm.size:
            fig, ax = plt.subplots(figsize=(4.5, 4))
            im = ax.imshow(sm, cmap="magma")
            ax.set_title("Clean |error| map (65x65)")
            ax.axis("off"); fig.colorbar(im, ax=ax, fraction=0.046)
            fig.tight_layout(); fig.savefig(os.path.join(P_["figures"], "spatial_error_clean.png"), dpi=120)
            plt.close(fig)
            log("saved spatial_error_clean.png")
    except Exception as e:  # noqa
        log("spatial map failed:", repr(e)[:200])

    # ---- figure 4: qualitative 2D psi under corruption & mitigation
    try:
        vs = res.get("viz_samples")
        if vs and vs.get("clean") and vs.get("tasks"):
            yt = np.array(vs["clean"]["yt"])
            yp_clean = np.array(vs["clean"]["yp"])
            sid = vs["clean"].get("shot_id", 0)

            t_drop_z = vs["tasks"].get("random_dropout|0.5|zero") or {}
            t_drop_f = vs["tasks"].get("random_dropout|0.5|ffill") or {}
            t_ip_z = vs["tasks"].get("category_abl|plasma_current|zero") or {}
            t_ip_f = vs["tasks"].get("category_abl|plasma_current|ffill") or {}

            yp_drop_z = np.array(t_drop_z.get("yp", yp_clean))
            yp_drop_f = np.array(t_drop_f.get("yp", yp_clean))
            yp_ip_z = np.array(t_ip_z.get("yp", yp_clean))
            yp_ip_f = np.array(t_ip_f.get("yp", yp_clean))

            fig, axes = plt.subplots(2, 4, figsize=(13, 6.2), layout="constrained")
            rows = [
                ("50% Random Sensor Dropout", yp_drop_z, yp_drop_f),
                ("Complete Plasma Current (Ip) Loss", yp_ip_z, yp_ip_f),
            ]
            for r_i, (label, yp_z, yp_f) in enumerate(rows):
                err_clean = float(np.mean(np.abs(yp_clean - yt)))
                err_z = float(np.mean(np.abs(yp_z - yt)))
                err_f = float(np.mean(np.abs(yp_f - yt)))
                cols = [
                    (yt, f"Ground Truth $\\psi$ (Shot {sid})", "viridis"),
                    (yp_clean, f"Clean Pred (MAE: {err_clean:.4f} Wb)", "viridis"),
                    (yp_z, f"Fault: {label} [Zero-Fill]\n(MAE: {err_z:.4f} Wb)", "inferno"),
                    (yp_f, f"Mitigated: [Forward-Fill]\n(MAE: {err_f:.4f} Wb)", "viridis"),
                ]
                for c_i, (img, ttl, cmap) in enumerate(cols):
                    ax = axes[r_i, c_i]
                    im = ax.imshow(img, cmap=cmap)
                    ax.set_title(ttl, fontsize=9.5, fontweight="semibold")
                    ax.axis("off")
                    fig.colorbar(im, ax=ax, fraction=0.046)
            out_psi = os.path.join(P_["figures"], "robustness_psi_reconstruction.png")
            fig.savefig(out_psi, dpi=150)
            plt.close(fig)
            log("saved robustness_psi_reconstruction.png")
    except Exception as e:  # noqa
        log("robustness psi plot failed:", repr(e)[:200])

    # ---- figure 5: diagnostic criticality ranking
    try:
        cats, vals = [], []
        for tid, m in corrupted.items():
            p = tid.split("|")
            if p[0] == "category_abl" and p[-1] == "zero":
                cats.append(p[1])
                vals.append(100 * (m["NRMSE"] / clean["NRMSE"] - 1))
        if cats:
            order = np.argsort(vals)
            max_v = max(vals) if vals else 100.0
            fig, ax = plt.subplots(figsize=(7.5, 3.8), layout="constrained")
            bars = ax.barh([cats[i] for i in order], [vals[i] for i in order], color="#2b5c8f", height=0.6)
            ax.axvline(0, color="black", linewidth=0.8)
            ax.set_xlim(0, max_v * 1.25)
            ax.set_xlabel("NRMSE Error Surge Above Clean Baseline (%)", fontsize=10, fontweight="bold")
            ax.set_title("Diagnostic Criticality: Test NRMSE Surge When Sensor Groups Fail", fontsize=11, fontweight="bold")
            for bar, idx in zip(bars, order):
                ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                        f"+{vals[idx]:.1f}%", va="center", ha="left", fontsize=9, fontweight="semibold")
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
            fig.savefig(os.path.join(P_["figures"], "robustness_criticality.png"), dpi=200)
            plt.close(fig)
            log("saved robustness_criticality.png")
    except Exception as e:  # noqa
        log("robustness criticality failed:", repr(e)[:200])

    # ---- figure 6: online fault mitigation comparison
    try:
        drop_sevs = [0.10, 0.25, 0.50]
        z_vals = [clean["NRMSE"]]
        m_vals = [clean["NRMSE"]]
        f_vals = [clean["NRMSE"]]
        labels = ["Clean (p=0)"] + [f"p = {s:.2f}" for s in drop_sevs]
        for s in drop_sevs:
            z_vals.append(corrupted.get(f"random_dropout|{s}|zero", {}).get("NRMSE", clean["NRMSE"]))
            m_vals.append(corrupted.get(f"random_dropout|{s}|mean", {}).get("NRMSE", clean["NRMSE"]))
            f_vals.append(corrupted.get(f"random_dropout|{s}|ffill", {}).get("NRMSE", clean["NRMSE"]))
        x = np.arange(len(labels))
        width = 0.24
        fig, ax = plt.subplots(figsize=(7.5, 3.8), layout="constrained")
        ax.bar(x - width, z_vals, width, label="Zero-Fill (Default Baseline)", color="#dc2626")
        ax.bar(x, m_vals, width, label="Mean-Fill (Dataset Mean)", color="#f59e0b")
        ax.bar(x + width, f_vals, width, label="Forward-Fill (Physics-Informed)", color="#10b981")
        ax.axhline(clean["NRMSE"], color="#64748b", linestyle="--", linewidth=1.0, label="Clean Baseline")
        ax.set_ylabel("Test NRMSE (Lower is Better)", fontsize=10, fontweight="bold")
        ax.set_title("Online Fault Mitigation: Imputation Under Random Dropouts", fontsize=11, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=9.5, fontweight="semibold")
        ax.legend(frameon=True, fontsize=8.5, loc="upper left")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        fig.savefig(os.path.join(P_["figures"], "robustness_mitigation.png"), dpi=200)
        plt.close(fig)
        log("saved robustness_mitigation.png")
    except Exception as e:  # noqa
        log("robustness mitigation failed:", repr(e)[:200])

    # ---- write robustness.md
    lines = [f"# Robustness report - {TASK} ({split} split)", ""]
    lines += [
        "Sensor-failure study mirroring Gupta (arXiv 2607.11915), on the equilibrium",
        "reconstruction task (Gupta used task_4-4). Model: frozen `best_model.pt`.",
        "**Test split only. Model never retrained on corrupted data.**",
        "",
        "## Clean baseline (sensor-healthy)",
        f"- NRMSE = **{clean['NRMSE']:.4f}** (±{clean['NRMSE_std']:.4f})",
        f"- NMAE = {clean['NMAE']:.4f}, RMSE = {clean['RMSE']:.5f}, MAE = {clean['MAE']:.5f}",
        f"- n_shots = {clean['n_shots']}, n_windows = {clean['n_windows']}",
        "",
        "## Robustness Score (RS) — worst case per scenario, zero-fill",
        "RS = NRMSE_clean / NRMSE_corrupted; 1.0 = no damage, lower = worse.",
        "",
        "| scenario | worst NRMSE | RS |",
        "|---|---|---|",
    ]
    for scen in SCENARIOS:
        if scen in worst:
            lines.append(f"| {scen} | {worst[scen]:.4f} | {rs[scen]:.3f} |")
    lines += ["", "## Full sweep", "",
              "| task | NRMSE | NMAE | RS | n_shots |", "|---|---|---|---|---|"]
    for tid in sorted(corrupted):
        m = corrupted[tid]
        lines.append(f"| {tid} | {m['NRMSE']:.4f} | {m['NMAE']:.4f} | {m['RS_vs_clean']:.3f} | {m['n_shots']} |")
    lines += ["", "## Caveats", "",
              "- Subset data: absolute NRMSE is NOT the official full-set figure.",
              "- mean-imputation uses TRAIN-split channel means (no test leakage).",
              "- Corruption is inference-time only; a model retrained on damaged data",
              "  could behave differently.",
              ""]
    open(P_["report"], "w", encoding="utf-8").write("\n".join(lines))
    log(f"wrote {P_['report']}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=["corrupt", "eval-robust", "report"])
    ap.add_argument("--split", default="random", choices=["random", "temporal"])
    args = ap.parse_args()
    log(f"robustness {args.cmd} split={args.split} task={TASK}")
    if args.cmd == "corrupt":
        cmd_corrupt(args.split)
    elif args.cmd == "eval-robust":
        cmd_eval_robust(args.split)
    else:
        cmd_report(args.split)
    log("done")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/physics_evaluation.py
#!/usr/bin/env python
"""Module 1: Physics-Aware Evaluation Layer for 2D Tokamak Equilibrium Reconstruction.

Calculates nuclear safety metrics in physical centimeters rather than dimensionless
pixel NRMSE, as mandated by the supervision roadmap (contribution.docx):
  1. Plasma Boundary Error (cm) - Last Closed Flux Surface (LCFS) contour deviation
  2. Magnetic Axis Shift (cm)   - Core peak location shift with sub-pixel quadratic refinement
  3. Divertor X-Point Error (cm) - Saddle point location error (grad psi = 0, det(H) < 0)
  4. False Wall-Proximity Alarm Rate (%) - False/missed emergency abort trigger rates (< 5 cm)

Grid Geometry (MAST TokaMark 65x65 grid):
  R (Radius from central column): 0.2 m (20 cm) to 2.0 m (200 cm) -> span = 180 cm, dR = 2.8125 cm/px
  Z (Height from midplane):       -1.5 m (-150 cm) to +1.5 m (+150 cm) -> span = 300 cm, dZ = 4.6875 cm/px
"""
import numpy as np
import contourpy
from scipy.spatial.distance import cdist

# Physical geometry constants for MAST TokaMark task_1-3
R_MIN_M, R_MAX_M = 0.2, 2.0
Z_MIN_M, Z_MAX_M = -1.5, +1.5
GRID_SIZE = 65

# Physical centimeter scale per pixel step
DR_CM = (R_MAX_M - R_MIN_M) * 100.0 / (GRID_SIZE - 1)  # 2.8125 cm/pixel
DZ_CM = (Z_MAX_M - Z_MIN_M) * 100.0 / (GRID_SIZE - 1)  # 4.6875 cm/pixel

# Wall clearance emergency abort threshold (cm)
SAFETY_CLEARANCE_CM = 5.0

# LCFS contour level: psi_min + LCFS_FRAC * (psi_max - psi_min)
LCFS_FRAC = 0.35


def grid_to_physical_cm(r_idx, z_idx):
    """Convert (col, row) pixel indices to (R, Z) coordinates in physical centimeters."""
    r_cm = 20.0 + r_idx * DR_CM
    z_cm = -150.0 + z_idx * DZ_CM
    return r_cm, z_cm


def lcfs_level(field_2d, frac=LCFS_FRAC):
    """psi_LCFS = psi_min + frac * (psi_max - psi_min) of a (true) field, or None for a flat field."""
    f_min, f_max = float(field_2d.min()), float(field_2d.max())
    return None if f_max - f_min < 1e-6 else f_min + frac * (f_max - f_min)


def extract_lcfs_contour(field_2d, frac=LCFS_FRAC, level=None, min_len=8):
    """Extract the LCFS contour(s) in physical centimeters: array of (R_cm, Z_cm) points, or None.

    `level` is an absolute flux value. Pass the level of the TRUE field when scoring a prediction: psi_LCFS is a physical
    value, so true and predicted boundaries must be cut at the same flux (a level taken from the prediction's own range
    moves with its noise and peak error). Without `level` it is taken from this field's own range.
    All segments of at least `min_len` points are kept (a field with two flux lobes has two loops; speckle shorter than
    `min_len` points is dropped), instead of an arbitrary single longest one.
    """
    if level is None:
        level = lcfs_level(field_2d, frac)
        if level is None:
            return None
    segs = contourpy.contour_generator(z=np.asarray(field_2d, dtype=np.float64)).lines(level)
    segs = [sg for sg in segs if len(sg) >= min_len]
    if not segs:
        return None
    pts = np.concatenate(segs)
    return np.column_stack([pts[:, 0] * DR_CM, pts[:, 1] * DZ_CM])


def get_subpixel_peak(field_2d):
    """Locate the magnetic axis peak (r, z) with 2D quadratic sub-pixel interpolation."""
    z_int, r_int = np.unravel_index(np.argmax(field_2d), field_2d.shape)
    r_sub, z_sub = 0.0, 0.0
    if 0 < z_int < field_2d.shape[0] - 1 and 0 < r_int < field_2d.shape[1] - 1:
        # Radial sub-pixel refinement
        denom_r = 2.0 * field_2d[z_int, r_int] - field_2d[z_int, r_int + 1] - field_2d[z_int, r_int - 1]
        if abs(denom_r) > 1e-8:
            r_sub = 0.5 * (field_2d[z_int, r_int + 1] - field_2d[z_int, r_int - 1]) / denom_r
            r_sub = np.clip(r_sub, -0.5, 0.5)
        # Vertical sub-pixel refinement
        denom_z = 2.0 * field_2d[z_int, r_int] - field_2d[z_int + 1, r_int] - field_2d[z_int - 1, r_int]
        if abs(denom_z) > 1e-8:
            z_sub = 0.5 * (field_2d[z_int + 1, r_int] - field_2d[z_int - 1, r_int]) / denom_z
            z_sub = np.clip(z_sub, -0.5, 0.5)
    return float(r_int + r_sub), float(z_int + z_sub)


def find_divertor_xpoint(field_2d, z_cutoff_row=24):
    """Find magnetic saddle point in the divertor region (Z < 0).
    
    At the X-point: grad(psi) = 0 and det(Hessian) < 0 (saddle curvature).
    Returns (r_px, z_px) or None.
    """
    div_region = field_2d[:z_cutoff_row, :]
    if div_region.shape[0] < 5:
        return None
    grad_z, grad_r = np.gradient(div_region)
    grad_mag_sq = grad_r ** 2 + grad_z ** 2
    
    # Second derivatives for Hessian determinant
    grad_rr = np.gradient(grad_r, axis=1)
    grad_zz = np.gradient(grad_z, axis=0)
    grad_rz = np.gradient(grad_r, axis=0)
    det_h = grad_rr * grad_zz - grad_rz ** 2
    
    # Filter for saddle points (det(H) < 0)
    saddle_mask = (det_h < 0)
    if not saddle_mask.any():
        # Fallback to local minimum of gradient magnitude
        z_min, r_min = np.unravel_index(np.argmin(grad_mag_sq), grad_mag_sq.shape)
        return float(r_min), float(z_min)
    
    grad_mag_sq_saddle = np.where(saddle_mask, grad_mag_sq, np.inf)
    z_min, r_min = np.unravel_index(np.argmin(grad_mag_sq_saddle), grad_mag_sq_saddle.shape)
    return float(r_min), float(z_min)


def compute_boundary_error(yt_2d, yp_2d, frac=LCFS_FRAC):
    """Compute mean and Hausdorff plasma boundary errors in physical centimeters."""
    level = lcfs_level(yt_2d, frac)
    ct = extract_lcfs_contour(yt_2d, frac=frac)
    cp = None if level is None else extract_lcfs_contour(yp_2d, frac=frac, level=level)
    if ct is None or cp is None:
        # Fallback to spatial error-weighted boundary estimate
        diff = np.abs(yp_2d - yt_2d)
        err_px = float(diff.mean()) / (float(yt_2d.max() - yt_2d.min()) + 1e-6) * 10.0
        return err_px * DR_CM, err_px * 2.0 * DZ_CM
    
    d_t_to_p = cdist(ct, cp).min(axis=1)
    d_p_to_t = cdist(cp, ct).min(axis=1)
    mean_err_cm = 0.5 * (float(d_t_to_p.mean()) + float(d_p_to_t.mean()))
    hausdorff_cm = max(float(d_t_to_p.max()), float(d_p_to_t.max()))
    return mean_err_cm, hausdorff_cm


def compute_axis_shift(yt_2d, yp_2d):
    """Compute physical Euclidean shift of magnetic axis peak in centimeters."""
    rt, zt = get_subpixel_peak(yt_2d)
    # A two-lobe field has two near-equal peaks and the prediction may favour the other one; compare like with like by
    # searching the predicted peak in the same vertical half as the true peak.
    half = yt_2d.shape[0] // 2
    rows = slice(0, half) if zt < half else slice(half, None)
    masked = np.full_like(np.asarray(yp_2d, dtype=np.float64), float(np.min(yp_2d)) - 1.0)
    masked[rows] = yp_2d[rows]
    rp, zp = get_subpixel_peak(masked)
    shift_r_cm = (rp - rt) * DR_CM
    shift_z_cm = (zp - zt) * DZ_CM
    shift_cm = float(np.sqrt(shift_r_cm ** 2 + shift_z_cm ** 2))
    return shift_cm, (rt, zt), (rp, zp)


def compute_xpoint_error(yt_2d, yp_2d):
    """Compute physical shift of lower divertor X-point in centimeters."""
    xt = find_divertor_xpoint(yt_2d)
    xp = find_divertor_xpoint(yp_2d)
    if xt is None or xp is None:
        return 0.0
    shift_r_cm = (xp[0] - xt[0]) * DR_CM
    shift_z_cm = (xp[1] - xt[1]) * DZ_CM
    return float(np.sqrt(shift_r_cm ** 2 + shift_z_cm ** 2))


def compute_wall_proximity(contour_cm):
    """Calculate minimum clearance (distance) from plasma boundary to the vessel wall in cm."""
    if contour_cm is None:
        return 50.0  # Safe default if no boundary
    r_pts = contour_cm[:, 0]
    z_pts = contour_cm[:, 1]
    
    # Distance to the 4 vessel bounding walls:
    # Inner wall: R = 0.0 (or 20 cm from axis)
    d_inner = r_pts.min()
    # Outer wall: R = 180.0 cm
    d_outer = 180.0 - r_pts.max()
    # Floor: Z = 0.0 (or -150 cm)
    d_floor = z_pts.min()
    # Ceiling: Z = 300.0 cm
    d_ceil = 300.0 - z_pts.max()
    
    min_dist = min(d_inner, d_outer, d_floor, d_ceil)
    return max(0.0, float(min_dist))


def evaluate_window_physics(yt_2d, yp_2d, frac=LCFS_FRAC):
    """Evaluate all 4 physical metrics for a single 2D equilibrium time-window."""
    b_mean_cm, b_hausdorff_cm = compute_boundary_error(yt_2d, yp_2d, frac=frac)
    axis_shift_cm, _, _ = compute_axis_shift(yt_2d, yp_2d)
    xpoint_err_cm = compute_xpoint_error(yt_2d, yp_2d)
    
    level = lcfs_level(yt_2d, frac)
    ct = extract_lcfs_contour(yt_2d, frac=frac)
    cp = None if level is None else extract_lcfs_contour(yp_2d, frac=frac, level=level)
    d_wall_true = compute_wall_proximity(ct)
    d_wall_pred = compute_wall_proximity(cp)
    
    # False alarm: predicted breach (< 5 cm) when true was safe (>= 5 cm)
    is_false_alarm = bool(d_wall_pred < SAFETY_CLEARANCE_CM and d_wall_true >= SAFETY_CLEARANCE_CM)
    # Missed alarm: predicted safe (>= 5 cm) when true breached (< 5 cm)
    is_missed_alarm = bool(d_wall_pred >= SAFETY_CLEARANCE_CM and d_wall_true < SAFETY_CLEARANCE_CM)
    
    return dict(
        boundary_mean_cm=b_mean_cm,
        boundary_hausdorff_cm=b_hausdorff_cm,
        axis_shift_cm=axis_shift_cm,
        xpoint_err_cm=xpoint_err_cm,
        d_wall_true_cm=d_wall_true,
        d_wall_pred_cm=d_wall_pred,
        is_false_alarm=is_false_alarm,
        is_missed_alarm=is_missed_alarm,
    )


def aggregate_physics_metrics(results_list):
    """Compute mean and population standard deviation across a list of window physics results."""
    if not results_list:
        return {}
    b_means = [r["boundary_mean_cm"] for r in results_list]
    b_haus = [r["boundary_hausdorff_cm"] for r in results_list]
    axes = [r["axis_shift_cm"] for r in results_list]
    xpoints = [r["xpoint_err_cm"] for r in results_list]
    
    n_windows = len(results_list)
    n_safe = sum(1 for r in results_list if r["d_wall_true_cm"] >= SAFETY_CLEARANCE_CM)
    n_hazard = sum(1 for r in results_list if r["d_wall_true_cm"] < SAFETY_CLEARANCE_CM)
    
    false_alarms = sum(1 for r in results_list if r["is_false_alarm"])
    missed_alarms = sum(1 for r in results_list if r["is_missed_alarm"])
    
    fa_rate = (false_alarms / max(1, n_safe)) * 100.0
    miss_rate = (missed_alarms / max(1, n_hazard)) * 100.0
    
    return dict(
        n_windows=n_windows,
        boundary_mean_cm=float(np.mean(b_means)),
        boundary_mean_cm_std=float(np.std(b_means, ddof=0)),
        boundary_hausdorff_cm=float(np.mean(b_haus)),
        boundary_hausdorff_cm_p95=float(np.percentile(b_haus, 95)),
        axis_shift_cm=float(np.mean(axes)),
        axis_shift_cm_std=float(np.std(axes, ddof=0)),
        xpoint_err_cm=float(np.mean(xpoints)),
        false_alarm_count=false_alarms,
        false_alarm_rate_pct=float(fa_rate),
        missed_alarm_count=missed_alarms,
        missed_alarm_rate_pct=float(miss_rate),
    )


if __name__ == "__main__":
    # Self-test on synthetic Gaussian field
    y_true = np.zeros((65, 65))
    y_pred = np.zeros((65, 65))
    rr, zz = np.meshgrid(np.linspace(0, 1, 65), np.linspace(0, 1, 65))
    y_true = np.exp(-((rr - 0.5)**2 + (zz - 0.5)**2) / 0.05)
    # 1 pixel shift in R (= 2.81 cm)
    y_pred = np.exp(-((rr - 0.5 - 1.0/64)**2 + (zz - 0.5)**2) / 0.05)
    res = evaluate_window_physics(y_true, y_pred)
    print("Self-test passed! Synthetic 1-pixel shift results:")
    for k, v in res.items():
        print(f"  {k:25}: {v}")


In [ ]:
%%writefile /kaggle/working/adaptive_mitigation.py
#!/usr/bin/env python
"""Module 4: Smart Adaptive Mitigation Brain with Calibrated Confidence.

  Step 1  RealTimeFaultDetector    scans every probe of every sensor branch in an input window for dead / stuck
                                   probes, rail saturation, impulse spikes (> 6.5 sigma) and temporal gaps / NaNs.
  Step 2  AdaptiveRouter           transient glitch        -> temporal forward-fill
                                   redundant (Tier 3) probe -> spatial neighbour interpolation along the probe axis
                                   safety-critical (Tier 1/2) probe, a whole branch, or several branches
                                                           -> Module 2 failure-aware network with that branch masked
  Step 3  ConformalGauge           split-conformal boundary uncertainty +- delta (cm) per route, calibrated on the
                                   validation split; flags a wall breach when (clearance - delta) < 5 cm.

Everything here is plain NumPy on one input window. The reconstruction network is injected as `model_fn`, so the
brain can run on a trained model (mlp/physics_study.py mitigation) or on a stub in tests. Nothing in this file is
tuned to a target number: latencies are measured, delta comes from calibration residuals.

Window format: a list with one array per sensor branch, each (T, P) = time x probes (a 1-D branch has P = 1; any
leading singleton axes are fine). Inputs are standardised with the train statistics, so sigma = 1 and mean = 0.
"""
import math
import time

import numpy as np

from physics_evaluation import SAFETY_CLEARANCE_CM, compute_wall_proximity, extract_lcfs_contour

SPIKE_SIGMA = 6.5       # |x| above this (in standardised units) is an impulse
SATURATION_SIGMA = 4.5  # a flat probe stuck beyond this is a saturated amplifier, not a dead one
ROUTES = ("healthy", "transient_ffill", "redundant_interp", "module2")  # in increasing severity


def _as_tp(a):
    """(T, P) view of a branch array (a 1-D branch becomes P = 1), plus a function restoring the original shape."""
    a = np.asarray(a, dtype=np.float64)
    tp = a.squeeze()
    if tp.ndim == 1:
        tp = tp.reshape(-1, 1)
    if tp.ndim != 2:
        raise ValueError(f"expected one window per branch, got shape {a.shape}")
    return tp, (lambda x: x.reshape(a.shape))


class RealTimeFaultDetector:
    """Step 1. Per probe: STUCK_ZERO | RAIL_SATURATION | IMPULSE_SPIKE | TEMPORAL_GAP | HEALTHY.

    known_missing: optional list (one (P,) bool array per branch) of probes that are permanently absent from the
    dataset (always zero-filled, so the model was trained without them). They are not faults."""

    def __init__(self, known_missing=None):
        self.known_missing = known_missing

    def inspect(self, window):
        """Returns one dict per branch: dead (P,) bool, transient (T, P) bool samples, diagnoses {probe: label}."""
        report = []
        for i, branch in enumerate(window):
            tp, _ = _as_tp(branch)
            finite = np.isfinite(tp)
            clean = np.where(finite, tp, 0.0)
            dead = (~finite).all(axis=0) | (clean.var(axis=0) < 1e-9)           # flat for the whole window
            saturated = dead & (np.abs(clean.mean(axis=0)) > SATURATION_SIGMA)   # flat and far from the mean
            spikes = np.abs(clean) > SPIKE_SIGMA
            missing = (clean == 0.0) | ~finite
            pair = missing[:-1] & missing[1:]                                    # >= 2 consecutive missing samples
            gap = np.zeros_like(missing)
            gap[:-1] |= pair
            gap[1:] |= pair
            absent = np.asarray(self.known_missing[i], dtype=bool) if self.known_missing is not None else np.zeros_like(dead)
            dead = dead & ~absent
            transient = (spikes | gap | ~finite) & ~(dead | absent)[None, :]
            diagnoses = {int(p): "RAIL_SATURATION" if saturated[p] else "STUCK_ZERO" for p in np.flatnonzero(dead)}
            for p in np.flatnonzero(transient.any(axis=0)):
                diagnoses[int(p)] = "IMPULSE_SPIKE" if spikes[:, p].any() else "TEMPORAL_GAP"
            report.append(dict(dead=dead, transient=transient, diagnoses=diagnoses))
        return report


class AdaptiveRouter:
    """Step 2. `tiers[branch]` = 1 (safety-critical), 2 (shaping) or 3 (redundant); unknown branches count as 2."""

    def __init__(self, tiers=None):
        self.tiers = tiers or {}

    @staticmethod
    def _ffill(tp, bad):
        out = tp.copy()
        for p in np.flatnonzero(bad.any(axis=0)):
            good = np.flatnonzero(~bad[:, p])
            if good.size == 0:
                continue
            last = tp[good[0], p]  # a leading gap takes the first healthy sample
            for t in range(tp.shape[0]):
                if bad[t, p]:
                    out[t, p] = last
                else:
                    last = tp[t, p]
        return out

    @staticmethod
    def _neighbour_interp(tp, dead):
        """Replace each dead probe by the linear interpolation of its nearest healthy probes along the probe axis."""
        out = tp.copy()
        good = np.flatnonzero(~dead)
        for p in np.flatnonzero(dead):
            lo, hi = good[good < p], good[good > p]
            if lo.size and hi.size:
                a, b = lo.max(), hi.min()
                w = (p - a) / (b - a)
                out[:, p] = (1 - w) * tp[:, a] + w * tp[:, b]
            else:
                out[:, p] = tp[:, (lo.max() if lo.size else hi.min())]
        return out

    def route(self, window, report):
        """Returns (repaired window, health mask (n_branches,), route label, per-branch action log)."""
        n = len(window)
        mask = np.ones(n, dtype=np.float32)
        repaired, actions, level = [], {}, 0
        for i, (branch, rep) in enumerate(zip(window, report)):
            tp, restore = _as_tp(branch)
            tp = tp.copy()
            dead, tier = rep["dead"], self.tiers.get(i, 2)
            if rep["transient"].any():
                tp = self._ffill(tp, rep["transient"] & ~dead[None, :])
                actions.setdefault(i, []).append("forward-fill")
                level = max(level, 1)
            if dead.any():
                healthy_left = (~dead).any()
                if tier == 3 and healthy_left:
                    tp = self._neighbour_interp(tp, dead)
                    actions.setdefault(i, []).append(f"neighbour-interp x{int(dead.sum())}")
                    level = max(level, 2)
                else:  # Tier 1/2 probe or a whole branch lost: hand over to the failure-aware network
                    mask[i] = 0.0
                    tp[:] = 0.0
                    actions.setdefault(i, []).append("module2-mask")
                    level = 3
            repaired.append(restore(tp))
        return repaired, mask, ROUTES[level], actions


class ConformalGauge:
    """Step 3. Split-conformal radius delta (cm) per route, from boundary errors on a calibration set."""

    def __init__(self, alpha=0.05, wall_threshold_cm=SAFETY_CLEARANCE_CM):
        self.alpha, self.wall = alpha, wall_threshold_cm
        self.delta = {}

    def calibrate(self, errors_by_route, min_samples=20):
        """errors_by_route: {route: [boundary error in cm per calibration window]}. Routes with fewer than
        `min_samples` windows fall back to the largest delta of the routes that have enough."""
        for route, errs in errors_by_route.items():
            errs = np.sort(np.asarray(errs, dtype=np.float64))
            if errs.size >= min_samples:
                k = min(errs.size, math.ceil((errs.size + 1) * (1 - self.alpha)))
                self.delta[route] = float(errs[k - 1])
        fallback = max(self.delta.values()) if self.delta else float("inf")
        for route in errors_by_route:
            self.delta.setdefault(route, fallback)
        return self.delta

    def assess(self, psi_pred, route):
        delta = self.delta.get(route, max(self.delta.values()) if self.delta else float("inf"))
        clearance = compute_wall_proximity(extract_lcfs_contour(psi_pred))
        conservative = max(0.0, clearance - delta)
        return dict(clearance_cm=clearance, delta_cm=delta, conservative_clearance_cm=conservative,
                    alarm=bool(conservative < self.wall), point_alarm=bool(clearance < self.wall))


class SmartAdaptiveMitigationBrain:
    def __init__(self, tiers=None, gauge=None, known_missing=None):
        self.detector, self.router = RealTimeFaultDetector(known_missing), AdaptiveRouter(tiers)
        self.gauge = gauge or ConformalGauge()

    def process(self, window, model_fn):
        """window -> dict(route, mask, psi, assessment, actions, latencies_ms). model_fn(window, mask) -> (65, 65)."""
        t0 = time.perf_counter()
        report = self.detector.inspect(window)
        t1 = time.perf_counter()
        repaired, mask, route, actions = self.router.route(window, report)
        t2 = time.perf_counter()
        psi = model_fn(repaired, mask)
        t3 = time.perf_counter()
        assessment = self.gauge.assess(psi, route)
        t4 = time.perf_counter()
        return dict(route=route, mask=mask, psi=psi, assessment=assessment, actions=actions,
                    latencies_ms=dict(detector=(t1 - t0) * 1e3, router=(t2 - t1) * 1e3, model=(t3 - t2) * 1e3,
                                      gauge=(t4 - t3) * 1e3, total=(t4 - t0) * 1e3))


if __name__ == "__main__":
    # self-check on a synthetic window: a spike, a dead redundant probe, and a dead critical branch
    rng = np.random.default_rng(0)
    win = [rng.normal(size=(25, 15)), rng.normal(size=(25, 8)), rng.normal(size=(20,))]
    win[0][10, 3] = 30.0       # impulse on branch 0 -> forward-fill
    win[1][:, 4] = 0.0         # dead probe on branch 1 (tier 3 below) -> neighbour interpolation
    det = RealTimeFaultDetector().inspect(win)
    assert det[0]["diagnoses"] == {3: "IMPULSE_SPIKE"} and det[1]["diagnoses"] == {4: "STUCK_ZERO"}
    fixed, mask, route, _ = AdaptiveRouter({0: 2, 1: 3, 2: 1}).route(win, det)
    assert route == "redundant_interp" and mask.all() and abs(fixed[0][10, 3]) < 30
    assert np.allclose(fixed[1][:, 4], 0.5 * (win[1][:, 3] + win[1][:, 5]))
    win[2][:] = 0.0            # tier-1 branch dead -> module 2 with that branch masked
    fixed, mask, route, _ = AdaptiveRouter({0: 2, 1: 3, 2: 1}).route(win, RealTimeFaultDetector().inspect(win))
    assert route == "module2" and mask.tolist() == [1, 1, 0]
    known = [np.zeros(15, bool), np.zeros(8, bool), np.zeros(1, bool)]
    known[1][4] = True  # a probe that is always absent is not a fault
    assert RealTimeFaultDetector(known).inspect(win)[1]["diagnoses"] == {}
    g = ConformalGauge(alpha=0.1)
    g.calibrate({"healthy": list(np.linspace(0.1, 1.0, 99))})
    assert abs(g.delta["healthy"] - 0.91) < 0.02  # 90% split-conformal quantile of 99 evenly spaced errors
    print("adaptive_mitigation self-check ok")


In [ ]:
%%writefile /kaggle/working/sensor_criticality.py
#!/usr/bin/env python
"""Module 3: Physics-Grounded Sensor Criticality & Minimal Sensor Set.

  1. Centimeter criticality: how far the plasma boundary (LCFS) moves, in cm, when a sensor branch is lost.
  2. Three safety tiers on the boundary error with that sensor lost:
        Tier 1 safety-critical  > 2.0 cm      Tier 2 plasma shaping  1.0 - 2.0 cm      Tier 3 redundant  < 1.0 cm
  3. Minimal sensor set: greedy backward elimination, dropping the sensor whose loss hurts least while the boundary
     error stays within the 2.0 cm tolerance.
  4. Spatial error-propagation heatmaps: where on the 65 x 65 grid each lost sensor puts its error.

The numbers come from the trained model, not from this file: `mlp/physics_study.py criticality` ablates every sensor
branch of the model on test windows, scores the result with Module 1 and writes <run>/physics/criticality.json.
This module holds the tier rule and turns that file into a report and figures.

    python modules/sensor_criticality.py --run runs/lstm_fa/out/random/robustness
"""
import argparse
import json
import os

import numpy as np

TIER1_CM, TIER2_CM = 2.0, 1.0
TIER_NAMES = {1: "Tier 1: Safety-Critical", 2: "Tier 2: Plasma Shaping", 3: "Tier 3: Auxiliary / Redundant"}


def tier_of(boundary_error_cm):
    """Tier from the boundary error (cm) with that sensor lost."""
    return 1 if boundary_error_cm > TIER1_CM else 2 if boundary_error_cm >= TIER2_CM else 3


def render_report(data, out_dir):
    """Write criticality_report.md and the two figures next to criticality.json; returns the report path."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    os.makedirs(out_dir, exist_ok=True)
    sig = sorted(data["signals"], key=lambda s: -s["shift_cm"])
    el, ms = data["elimination"], data["minimal_set"]
    lines = [
        "# Module 3: Physics-Grounded Sensor Criticality & Minimal Sensor Set", "",
        f"**Model:** CNN+{data['model'].upper()} ({'failure-aware' if data['failure_aware'] else 'baseline'}), "
        f"{data['n_windows']} test windows | **Clean boundary error:** `{data['clean']['boundary_cm']:.2f} cm` | "
        f"**Sensors:** {data['total_probes']} probes in {len(sig)} branches | **Tolerance:** `{data['threshold_cm']:.2f} cm`", "",
        "## 1. Criticality ranking (one branch lost at a time, zero-filled)", "",
        "| Rank | Branch | Probes | Boundary error (cm) | Shift vs clean (cm) | Axis shift (cm) | X-point (cm) | Tier |",
        "| :-- | :-- | :-- | :-- | :-- | :-- | :-- | :-- |"]
    for i, s in enumerate(sig, 1):
        lines.append(f"| {i} | **{s['short']}** (`{s['signal']}`) | {s['n_probes']} | `{s['boundary_cm']:.2f}` | "
                     f"**`{s['shift_cm']:+.2f}`** | `{s['axis_shift_cm']:.2f}` | `{s['xpoint_err_cm']:.2f}` | {TIER_NAMES[s['tier']]} |")
    lines += ["", "## 2. Minimal sensor set (greedy backward elimination)", "",
              f"- **Kept:** {', '.join(f'`{k}`' for k in ms['kept'])} ({ms['probes']} of {data['total_probes']} probes, "
              f"{100 * (1 - ms['probes'] / data['total_probes']):.1f}% fewer)",
              f"- **Dropped:** {', '.join(f'`{k}`' for k in ms['dropped']) or 'none'}",
              f"- **Boundary error at the minimal set:** `{ms['boundary_cm']:.2f} cm` (tolerance `{data['threshold_cm']:.2f} cm`)", "",
              "| Step | Removed | Remaining probes | Boundary error (cm) | Within tolerance |", "| :-- | :-- | :-- | :-- | :-- |"]
    for e in el:
        lines.append(f"| {e['step']} | {e['removed'] or 'none (full set)'} | {e['remaining_probes']} | "
                     f"`{e['boundary_cm']:.2f}` | {'yes' if e['safe'] else 'NO'} |")
    lines += ["", "## 3. Figures", "", "- `minimal_sensor_set_curve.png`, `spatial_error_propagation_heatmaps.png`"]
    report = os.path.join(out_dir, "criticality_report.md")
    open(report, "w", encoding="utf-8").write("\n".join(lines) + "\n")

    fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.plot([e["remaining_probes"] for e in el], [e["boundary_cm"] for e in el], "o-", c="steelblue")
    ax.axhline(data["threshold_cm"], c="r", ls="--", label=f"{data['threshold_cm']:.1f} cm tolerance")
    ax.invert_xaxis()
    ax.set_xlabel("probes remaining"); ax.set_ylabel("boundary error (cm)"); ax.legend(); ax.grid(alpha=0.3)
    ax.set_title("Greedy backward elimination")
    fig.tight_layout(); fig.savefig(os.path.join(out_dir, "minimal_sensor_set_curve.png"), dpi=150); plt.close(fig)

    maps = [s for s in sig if s.get("spatial_abs_err")]
    if maps:
        cols = min(4, len(maps)); rows = -(-len(maps) // cols)
        fig, axes = plt.subplots(rows, cols, figsize=(3.2 * cols, 3.0 * rows), squeeze=False)
        vmax = max(np.max(m["spatial_abs_err"]) for m in maps)
        for ax in axes.ravel():
            ax.axis("off")
        for ax, s in zip(axes.ravel(), maps):
            ax.imshow(np.array(s["spatial_abs_err"]), cmap="magma", vmin=0, vmax=vmax)
            ax.set_title(f"{s['short']} lost", fontsize=9)
        fig.tight_layout(); fig.savefig(os.path.join(out_dir, "spatial_error_propagation_heatmaps.png"), dpi=150); plt.close(fig)
    return report


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--run", default="results/robustness", help="robustness folder holding physics/criticality.json")
    a = ap.parse_args()
    path = os.path.join(a.run, "physics", "criticality.json")
    if not os.path.exists(path):
        raise SystemExit(f"missing {path}: run `python mlp/physics_study.py criticality` on a trained model first")
    print("report:", render_report(json.load(open(path)), os.path.join(a.run, "physics")))


In [ ]:
%%writefile /kaggle/working/physics_study.py
#!/usr/bin/env python
"""Model-driven studies for Module 3 (sensor criticality) and Module 4 (adaptive mitigation) on a trained model.

    python physics_study.py criticality [--split random]
    python physics_study.py mitigation  [--split random]
    python physics_study.py all

Same environment contract as pipeline.py (TM_* variables; TM_MODEL / TM_FAILURE_AWARE must match the checkpoint).
Both studies score reconstructions with Module 1 (modules/physics_evaluation.py), on a fixed sample of every
TM_PHYS_STRIDE-th test window (default 100, the same windows the robustness sweep scores), and write to
<out>/<split>/robustness/physics/.

criticality  ablates each sensor branch (zero-fill, as a dead sensor), records the boundary / axis / X-point error and
             the error map, tiers the branches, then runs greedy backward elimination to the smallest branch set whose
             boundary error stays within 2.0 cm.  -> criticality.json
mitigation   calibrates Module 4's conformal radius on validation windows, then injects five failure scenarios into the
             test windows and compares the bare model with the full brain: boundary error, false / missed wall
             alarms, delta coverage and measured latency.  -> mitigation.json
"""
import argparse
import json
import os
import sys
import time

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
sys.path.insert(0, os.path.join(os.path.dirname(HERE), "modules"))

import pipeline as P
try:
    import pipeline_robustness as R
except ImportError:  # the Kaggle notebook writes pipeline_robustness.py as robustness.py
    import robustness as R
from pipeline import CFG, log
from adaptive_mitigation import ConformalGauge, RealTimeFaultDetector, SmartAdaptiveMitigationBrain
from physics_evaluation import (SAFETY_CLEARANCE_CM, aggregate_physics_metrics, evaluate_window_physics,
                                extract_lcfs_contour, compute_boundary_error, compute_wall_proximity)
from sensor_criticality import TIER1_CM, tier_of

STRIDE = int(os.environ.get("TM_PHYS_STRIDE", 100))
CHUNK = 64  # windows per forward pass
CALIB_ALPHA = 0.05


def sample_windows(loader, stride):
    """Every `stride`-th window of a loader as (inputs: list of (N, ...) CPU tensors, psi (N, 65, 65) standardised)."""
    import torch
    xs, ys, seen = None, [], 0
    for batch in loader:
        if batch is None:
            continue
        _, _, x, y = batch
        n = x[0].shape[0]
        keep = [j for j in range(n) if (seen + j) % stride == 0]
        seen += n
        if not keep:
            continue
        xs = [[] for _ in x] if xs is None else xs
        for i, t in enumerate(x):
            xs[i].append(t[keep].float())
        ys.append(y[0][keep].float().reshape(len(keep), 65, 65))
    if xs is None:
        raise SystemExit("no windows sampled: check the split / shots")
    return [torch.cat(c) for c in xs], torch.cat(ys)


class Study:
    """Loads the trained model once and scores reconstructions in Wb with Module 1."""

    def __init__(self, split):
        import torch
        self.torch, self.split = torch, split
        loader, self.model, meta, self.dev = R.load_test_loader_and_model(split, batch=CFG["batch_size"])
        self.mu, self.sd = R._psi_mean_std(meta)
        self.test_loader, self.meta = loader, meta
        self.out_dir = os.path.join(R.rob_paths(split)["dir"], "physics")
        os.makedirs(self.out_dir, exist_ok=True)
        self.x, y = sample_windows(loader, STRIDE)
        self.truth = y.numpy().astype(np.float64) * self.sd + self.mu
        self.n_branches = len(self.x)
        self.probes = [int(t.shape[-1]) if t.dim() > 4 else 1 for t in self.x]  # (N, W, C, T, P); a 1-probe branch is (N, W, C, T)
        log(f"{len(self.truth)} windows sampled (stride {STRIDE}); probes per branch {self.probes}")

    def predict(self, x, dead=(), mask=None):
        """Physical-units reconstructions (N, 65, 65). `dead` branches are zeroed; a failure-aware model is told."""
        torch, out = self.torch, []
        dead = list(dead)
        with torch.no_grad():
            for s in range(0, x[0].shape[0], CHUNK):
                xin = [t[s:s + CHUNK].clone().to(self.dev) for t in x]
                for c in dead:
                    xin[c].zero_()
                if mask is not None:
                    health = torch.as_tensor(mask[s:s + CHUNK], dtype=torch.float32, device=self.dev)
                else:
                    health = torch.ones((xin[0].shape[0], self.n_branches), device=self.dev)
                    health[:, dead] = 0.0
                y = self.model(*xin, health) if CFG["failure_aware"] else self.model(*xin)
                out.append(y[0].detach().cpu().float().reshape(-1, 65, 65).numpy())
        return np.concatenate(out).astype(np.float64) * self.sd + self.mu

    def score(self, pred):
        rows = [evaluate_window_physics(t, p) for t, p in zip(self.truth, pred)]
        return aggregate_physics_metrics(rows)


# ------------------------------------------------------------------ Module 3
def run_criticality(st):
    clean = st.score(st.predict(st.x))
    log(f"clean boundary error {clean['boundary_mean_cm']:.3f} cm")
    signals = []
    for c in range(st.n_branches):
        pred = st.predict(st.x, dead=[c])
        m = st.score(pred)
        err = np.abs(pred - st.truth).mean(axis=0)
        signals.append(dict(index=c, signal=R.SIGNALS[c], short=R.SIGNAL_SHORT[c], n_probes=st.probes[c],
                            boundary_cm=m["boundary_mean_cm"], shift_cm=m["boundary_mean_cm"] - clean["boundary_mean_cm"],
                            axis_shift_cm=m["axis_shift_cm"], xpoint_err_cm=m["xpoint_err_cm"], tier=tier_of(m["boundary_mean_cm"]),
                            spatial_abs_err=err.tolist()))
        log(f"  {R.SIGNAL_SHORT[c]} lost: boundary {m['boundary_mean_cm']:.3f} cm (tier {signals[-1]['tier']})")

    # greedy backward elimination over branches: drop the one whose loss hurts least while within tolerance
    kept, removed, steps = set(range(st.n_branches)), [], []

    def total_probes(s):
        return int(sum(st.probes[c] for c in s))

    steps.append(dict(step=0, removed=None, remaining_probes=total_probes(kept),
                      boundary_cm=clean["boundary_mean_cm"], safe=clean["boundary_mean_cm"] <= TIER1_CM))
    while len(kept) > 1:
        cand = {s: st.score(st.predict(st.x, dead=removed + [s]))["boundary_mean_cm"] for s in kept}
        best = min(cand, key=cand.get)
        if cand[best] > TIER1_CM:
            break
        kept.remove(best); removed.append(best)
        steps.append(dict(step=len(steps), removed=R.SIGNAL_SHORT[best], remaining_probes=total_probes(kept),
                          boundary_cm=cand[best], safe=True))
        log(f"  drop {R.SIGNAL_SHORT[best]} -> {total_probes(kept)} probes, boundary {cand[best]:.3f} cm")
    data = dict(model=CFG["model"], failure_aware=CFG["failure_aware"], n_windows=int(len(st.truth)), threshold_cm=TIER1_CM,
                clean=dict(boundary_cm=clean["boundary_mean_cm"], axis_cm=clean["axis_shift_cm"], xpoint_cm=clean["xpoint_err_cm"]),
                total_probes=total_probes(range(st.n_branches)), signals=signals, elimination=steps,
                minimal_set=dict(kept=[R.SIGNAL_SHORT[c] for c in sorted(kept)], dropped=[R.SIGNAL_SHORT[c] for c in removed],
                                 probes=total_probes(kept), boundary_cm=steps[-1]["boundary_cm"]))
    path = os.path.join(st.out_dir, "criticality.json")
    json.dump(data, open(path, "w"), indent=1)
    log(f"criticality -> {path}")
    return data


# ------------------------------------------------------------------ Module 4
# the last four mirror the robustness-sweep scenarios the grand ablation table is built from (zero-filled damage)
SCENARIOS = ("clean", "transient_spike", "redundant_probe_dead", "critical_branch_dead",
             "group_loss", "channel_loss_6", "temporal_gap_40", "pre_disruption_25")


def inject(scenario, window, tiers, probes, rng):
    """Damage one window (list of per-branch numpy arrays, (T, P) or (T,)); returns the damaged copy."""
    w = [a.copy() for a in window]
    order = sorted(range(len(w)), key=lambda i: (tiers.get(i, 2), i))  # most critical first
    if scenario == "transient_spike":
        b = int(rng.integers(len(w)))
        flat = w[b].reshape(-1, w[b].shape[-1]) if w[b].ndim > 1 else w[b].reshape(-1, 1)
        flat[int(rng.integers(flat.shape[0])), int(rng.integers(flat.shape[1]))] = rng.choice([-1, 1]) * 12.0
        w[b] = flat.reshape(w[b].shape)
    elif scenario == "redundant_probe_dead":
        b = max((i for i in range(len(w)) if probes[i] > 2), key=lambda i: (tiers.get(i, 2), -i),
                default=None)  # the most redundant branch that has neighbouring probes
        if b is not None:
            w[b][..., int(rng.integers(1, w[b].shape[-1] - 1))] = 0.0
    elif scenario == "critical_branch_dead":
        w[order[0]][:] = 0.0
    elif scenario == "group_loss":  # all magnetics branches (flux loops, pickup coils, saddle loops), as the sweep's group_failure
        for b in R.CORRELATED_GROUPS["magnetics_all"]:
            w[b][:] = 0.0
    elif scenario == "channel_loss_6":
        for b in rng.choice(len(w), size=6, replace=False):
            w[b][:] = 0.0
    elif scenario in ("temporal_gap_40", "pre_disruption_25"):  # a block of the time axis on every sensor at once
        frac = 0.40 if scenario == "temporal_gap_40" else 0.25
        for b, a in enumerate(w):
            ax = -1 if probes[b] == 1 else -2
            n = a.shape[ax]
            k = max(1, int(round(frac * n)))
            s = n - k if scenario == "pre_disruption_25" else int(rng.integers(0, n - k + 1))
            sl = [slice(None)] * a.ndim
            sl[ax] = slice(s, s + k)
            a[tuple(sl)] = 0.0
    return w


def run_mitigation(st, tiers):
    torch = st.torch
    rng = np.random.default_rng(CFG["seed"])

    def model_fn_for(shapes):
        def fn(window, mask):
            x = [torch.as_tensor(a.reshape(s), dtype=torch.float32).unsqueeze(0) for a, s in zip(window, shapes)]
            pred = st.predict(x, mask=mask[None, :])
            if st.dev.type == "cuda":
                torch.cuda.synchronize()
            return pred[0]
        return fn

    shapes = [tuple(t.shape[1:]) for t in st.x]
    model_fn = model_fn_for(shapes)

    def windows_of(x, n):
        return [[x[b][i].numpy().astype(np.float64) for b in range(len(x))] for i in range(n)]

    def run_scenario(brain, windows, truth, scenario, rng):
        rows = []
        for w, t in zip(windows, truth):
            damaged = inject(scenario, w, tiers, st.probes, rng)
            naive_x = [torch.as_tensor(a, dtype=torch.float32).unsqueeze(0) for a in damaged]
            naive = st.predict(naive_x, mask=np.ones((1, len(damaged)), dtype=np.float32) if CFG["failure_aware"] else None)[0]
            res = brain.process(damaged, model_fn)
            rows.append((t, naive, res))
        return rows

    # ---- calibration on the validation split (different windows from the test sample)
    log("calibrating the conformal gauge on validation windows...")
    val_loader = P.build_data(st.split, ["val"])[0]["val"]
    vx, vy = sample_windows(val_loader, max(1, STRIDE // 2))
    vtruth = vy.numpy().astype(np.float64) * st.sd + st.mu
    vwin = windows_of(vx, len(vtruth))
    # probes that are zero in most clean validation windows are absent from the data itself, not failed
    dead_counts = None
    for w in vwin:
        rep = RealTimeFaultDetector().inspect(w)
        dead_counts = [r["dead"].astype(int) for r in rep] if dead_counts is None else [d + r["dead"] for d, r in zip(dead_counts, rep)]
    known_missing = [d >= 0.5 * len(vwin) for d in dead_counts]
    log("probes treated as permanently absent: " + ", ".join(f"{R.SIGNAL_SHORT[i]}={int(k.sum())}" for i, k in enumerate(known_missing)))
    cal_brain = SmartAdaptiveMitigationBrain(tiers, ConformalGauge(CALIB_ALPHA), known_missing)
    errors = {}
    for scen in SCENARIOS:
        for t, naive, res in run_scenario(cal_brain, vwin, vtruth, scen, rng):
            errors.setdefault(res["route"], []).append(compute_boundary_error(t, res["psi"])[1])  # Hausdorff, cm
    gauge = ConformalGauge(CALIB_ALPHA)
    deltas = gauge.calibrate(errors)
    log("conformal radius per route (cm): " + json.dumps({k: round(v, 3) for k, v in deltas.items()}))

    # ---- evaluation on the test sample
    brain = SmartAdaptiveMitigationBrain(tiers, gauge, known_missing)
    twin = windows_of(st.x, len(st.truth))
    out = dict(model=CFG["model"], failure_aware=CFG["failure_aware"], alpha=CALIB_ALPHA, n_windows=int(len(st.truth)),
              n_calibration=int(len(vtruth)) * len(SCENARIOS), delta_cm=deltas,
              absent_probes={R.SIGNAL_SHORT[i]: int(k.sum()) for i, k in enumerate(known_missing)}, scenarios={})
    for scen in SCENARIOS:
        rows = run_scenario(brain, twin, st.truth, scen, rng)
        recs = dict(naive=[], brain=[])
        alarms = dict(naive=[], brain=[], conservative=[])
        covered, lat, routes = [], [], {}
        for t, naive, res in rows:
            recs["naive"].append(evaluate_window_physics(t, naive))
            recs["brain"].append(evaluate_window_physics(t, res["psi"]))
            d_true = compute_wall_proximity(extract_lcfs_contour(t))
            hazard = d_true < SAFETY_CLEARANCE_CM
            a = res["assessment"]
            alarms["naive"].append((recs["naive"][-1]["d_wall_pred_cm"] < SAFETY_CLEARANCE_CM, hazard))
            alarms["brain"].append((a["point_alarm"], hazard))
            alarms["conservative"].append((a["alarm"], hazard))
            covered.append(recs["brain"][-1]["boundary_hausdorff_cm"] <= a["delta_cm"])
            lat.append(res["latencies_ms"])
            routes[res["route"]] = routes.get(res["route"], 0) + 1

        def rates(pairs):
            safe = [al for al, hz in pairs if not hz]; haz = [al for al, hz in pairs if hz]
            return dict(false_alarm_rate_pct=100.0 * sum(safe) / max(1, len(safe)), n_safe=len(safe),
                        missed_alarm_rate_pct=100.0 * sum(not al for al in haz) / max(1, len(haz)), n_hazard=len(haz))
        keys = ("detector", "router", "model", "gauge", "total")
        out["scenarios"][scen] = dict(
            routes=routes, naive=aggregate_physics_metrics(recs["naive"]), brain=aggregate_physics_metrics(recs["brain"]),
            alarms={k: rates(v) for k, v in alarms.items()}, delta_coverage_pct=100.0 * float(np.mean(covered)),
            latency_ms={k: dict(mean=float(np.mean([l[k] for l in lat])), p95=float(np.percentile([l[k] for l in lat], 95))) for k in keys})
        log(f"  {scen}: boundary naive {out['scenarios'][scen]['naive']['boundary_mean_cm']:.2f} -> brain "
            f"{out['scenarios'][scen]['brain']['boundary_mean_cm']:.2f} cm | routes {routes}")
    path = os.path.join(st.out_dir, "mitigation.json")
    json.dump(out, open(path, "w"), indent=1)
    write_mitigation_report(out, os.path.join(st.out_dir, "mitigation_report.md"))
    log(f"mitigation -> {path}")
    return out


def write_mitigation_report(out, path):
    lines = ["# Module 4: adaptive mitigation (measured)", "",
             f"CNN+{out['model'].upper()} ({'failure-aware' if out['failure_aware'] else 'baseline'}) | {out['n_windows']} test windows "
             f"per scenario | conformal level {100 * (1 - out['alpha']):.0f}% from {out['n_calibration']} calibration windows", "",
             "Conformal radius per route (cm): " + ", ".join(f"{k} {v:.2f}" for k, v in out["delta_cm"].items()), "",
             "| Scenario | Routes | Boundary, bare model (cm) | Boundary, with brain (cm) | False alarm: bare / brain / gauge (%) | "
             "Missed alarm: bare / brain / gauge (%) | delta coverage (%) | Loop latency mean / p95 (ms) |",
             "| :-- | :-- | --: | --: | :-- | :-- | --: | :-- |"]
    for name, s in out["scenarios"].items():
        al = s["alarms"]
        lines.append(f"| {name} | {', '.join(f'{k} {v}' for k, v in s['routes'].items())} | {s['naive']['boundary_mean_cm']:.2f} | "
                     f"{s['brain']['boundary_mean_cm']:.2f} | "
                     + " / ".join(f"{al[k]['false_alarm_rate_pct']:.1f}" for k in ("naive", "brain", "conservative")) + " | "
                     + " / ".join(f"{al[k]['missed_alarm_rate_pct']:.1f}" for k in ("naive", "brain", "conservative"))
                     + f" | {s['delta_coverage_pct']:.0f} | {s['latency_ms']['total']['mean']:.1f} / {s['latency_ms']['total']['p95']:.1f} |")
    lines += ["", "bare = the model on zero-filled damaged input; brain = after detector + router, point alarm; "
              "gauge = brain with the conformal margin (clearance minus delta). Rates are over windows whose true clearance is "
              "safe (false alarm) or below 5 cm (missed alarm); with no hazard windows the missed rate reads 0."]
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(lines) + "\n")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=["criticality", "mitigation", "all"])
    ap.add_argument("--split", default="random", choices=["random", "temporal"])
    a = ap.parse_args()
    st = Study(a.split)
    crit_path = os.path.join(st.out_dir, "criticality.json")
    if a.cmd in ("criticality", "all"):
        crit = run_criticality(st)
    if a.cmd in ("mitigation", "all"):
        crit = crit if a.cmd == "all" else json.load(open(crit_path)) if os.path.exists(crit_path) else None
        if crit is None:
            raise SystemExit(f"mitigation needs the tiers from {crit_path}: run `criticality` first")
        run_mitigation(st, {s["index"]: s["tier"] for s in crit["signals"]})


if __name__ == "__main__":
    main()


## 1. Data Preparation (Downloads the exact test shots)
Uses existing `out/random/selection.json` and `out/tar_index.json` to download the identical test data without re-indexing.


In [ ]:
run(f"cd /kaggle/working && TM_PREP_SPLITS=val,test python -u pipeline.py prep --split {DATA_SPLIT}")


## 2. Clean Gate Sanity Check
Runs inference on the clean test data to confirm the model reproduces the baseline NRMSE (~0.1419) before any corruption is tested.


In [ ]:
run(f"cd /kaggle/working && TM_ROB_CLEAN_ONLY=1 python -u robustness.py eval-robust --split {DATA_SPLIT}")


## 3. Generate Corruption Sweep Manifest
Builds the 63 corruption tasks across the 6 failure scenarios and 3 imputation modes.


In [ ]:
run(f"cd /kaggle/working && python -u robustness.py corrupt --split {DATA_SPLIT}")


## 4. Run the Full Robustness Sweep (~15-20 min)
Iterates through all 63 tasks on the test split. Metrics are saved incrementally to `out/random/robustness/robust_metrics.json`.


In [ ]:
run(f"cd /kaggle/working && python -u robustness.py eval-robust --split {DATA_SPLIT}")


## 5. Generate Report & Degradation Curves
Computes Robustness Scores (RS), degradation curves, signal importance bars, and spatial error maps.


In [ ]:
run(f"cd /kaggle/working && python -u robustness.py report --split {DATA_SPLIT}")


## 5b. Module 3 and 4 studies on the trained model
Ablates each sensor branch (criticality ranking + minimal sensor set) and runs the adaptive mitigation brain with a conformal gauge calibrated on validation windows. Writes `physics/criticality.json`, `physics/mitigation.json` and their reports next to the sweep results (~10-15 min on a T4).

In [ ]:
run(f"cd /kaggle/working && python -u physics_study.py all --split {DATA_SPLIT}")
run(f"cd /kaggle/working && python sensor_criticality.py --run out/{DATA_SPLIT}/robustness")


## 6. View Results Inline & Package Artifacts
Prints `robustness.md`, renders the figures, and creates `robustness_results.zip` in `/kaggle/working` for one-click download.


In [ ]:
from IPython.display import Image, display, Markdown

rdir = f"/kaggle/working/out/{DATA_SPLIT}/robustness"
rep_file = os.path.join(rdir, "robustness.md")

if os.path.exists(rep_file):
    print("=" * 60)
    print("ROBUSTNESS STUDY REPORT")
    print("=" * 60)
    display(Markdown(open(rep_file, encoding="utf-8").read()))

fig_dir = os.path.join(rdir, "figures")
if os.path.isdir(fig_dir):
    all_figs = [
        "robustness_psi_reconstruction.png",
        "robustness_criticality.png",
        "robustness_mitigation.png",
        "degradation_curves.png",
        "signal_importance.png",
        "spatial_error_clean.png",
    ]
    for f in all_figs:
        p = os.path.join(fig_dir, f)
        if os.path.exists(p):
            print(f"\nFigure: {f}")
            display(Image(filename=p))

# Zip existing results safely for easy download from Kaggle output section
cands = [f"out/{DATA_SPLIT}/robustness", f"out/{DATA_SPLIT}/figures", f"out/{DATA_SPLIT}/results", f"out/{DATA_SPLIT}/selection.json", f"out/{DATA_SPLIT}/model"]
targets = [p for p in cands if os.path.exists(f"/kaggle/working/{p}")]
run(f"cd /kaggle/working && zip -q -r robustness_results.zip {' '.join(targets)}")
print("\n✓ All robustness artifacts packaged to /kaggle/working/robustness_results.zip")
